In [1]:
!pip install requests pandas ipython

In [2]:
# @title
# ============================================
# БЛОК 1: УСТАНОВКА И ИМПОРТ БИБЛИОТЕК
# ============================================

!pip install requests pandas python-dotenv googletrans==4.0.0-rc1

import requests
import json
import pandas as pd
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional
import time
from collections import defaultdict
from IPython.display import display, HTML
import os
from dotenv import load_dotenv
import random
import asyncio
from googletrans import Translator

# Загрузка переменных окружения
load_dotenv()

print("✅ Все библиотеки импортированы")

✅ Все библиотеки импортированы


ключи API:

news: 92cea5faf7800eaa4302c868bcb934ec

calendar: uLWjql0491TAhgt77einRqPhNSopFWqW

weather: 39c76ed23fa5174e59309ef9b6f3fb98

In [3]:
# @title
# ============================================
# БЛОК 2: БАЗОВЫЙ КЛАСС ДЛЯ РАБОТЫ С API
# ============================================

class RealNewsPredictor:
    """
    Класс для предсказания новостей с использованием реальных API
    Поддерживает 50 стран
    """

    def __init__(self, news_api_key: str, calendar_api_key: str, weather_api_key: str):
        self.news_api_key = news_api_key
        self.calendar_api_key = calendar_api_key
        self.weather_api_key = weather_api_key

        # Кэш для API запросов
        self.cache = {}

        # Базовые URL API
        self.news_api_url = "https://newsapi.org/v2"
        self.calendar_api_url = "https://calendarific.com/api/v2"
        self.weather_api_url = "https://api.openweathermap.org/data/2.5"

        # ==================== 50 СТРАН С ПОЛНЫМИ ДАННЫМИ ====================
        self.countries_data = {
            # Северная Америка
            "USA": {"code": "US", "lang": "en", "capital": "Washington, D.C.", "lat": 38.8951, "lon": -77.0364},
            "Canada": {"code": "CA", "lang": "en", "capital": "Ottawa", "lat": 45.4215, "lon": -75.6972},
            "Mexico": {"code": "MX", "lang": "es", "capital": "Mexico City", "lat": 19.4326, "lon": -99.1332},

            # Южная Америка
            "Argentina": {"code": "AR", "lang": "es", "capital": "Buenos Aires", "lat": -34.6037, "lon": -58.3816},
            "Brazil": {"code": "BR", "lang": "pt", "capital": "Brasília", "lat": -15.8267, "lon": -47.9218},
            "Chile": {"code": "CL", "lang": "es", "capital": "Santiago", "lat": -33.4489, "lon": -70.6693},
            "Colombia": {"code": "CO", "lang": "es", "capital": "Bogotá", "lat": 4.7110, "lon": -74.0721},
            "Peru": {"code": "PE", "lang": "es", "capital": "Lima", "lat": -12.0464, "lon": -77.0428},
            "Venezuela": {"code": "VE", "lang": "es", "capital": "Caracas", "lat": 10.4806, "lon": -66.9036},
            "Uruguay": {"code": "UY", "lang": "es", "capital": "Montevideo", "lat": -34.9011, "lon": -56.1645},

            # Европа
            "UK": {"code": "GB", "lang": "en", "capital": "London", "lat": 51.5074, "lon": -0.1278},
            "France": {"code": "FR", "lang": "fr", "capital": "Paris", "lat": 48.8566, "lon": 2.3522},
            "Germany": {"code": "DE", "lang": "de", "capital": "Berlin", "lat": 52.5200, "lon": 13.4050},
            "Italy": {"code": "IT", "lang": "it", "capital": "Rome", "lat": 41.9028, "lon": 12.4964},
            "Spain": {"code": "ES", "lang": "es", "capital": "Madrid", "lat": 40.4168, "lon": -3.7038},
            "Netherlands": {"code": "NL", "lang": "nl", "capital": "Amsterdam", "lat": 52.3676, "lon": 4.9041},
            "Belgium": {"code": "BE", "lang": "nl", "capital": "Brussels", "lat": 50.8503, "lon": 4.3517},
            "Switzerland": {"code": "CH", "lang": "de", "capital": "Bern", "lat": 46.9480, "lon": 7.4474},
            "Sweden": {"code": "SE", "lang": "sv", "capital": "Stockholm", "lat": 59.3293, "lon": 18.0686},
            "Norway": {"code": "NO", "lang": "no", "capital": "Oslo", "lat": 59.9139, "lon": 10.7522},
            "Denmark": {"code": "DK", "lang": "da", "capital": "Copenhagen", "lat": 55.6761, "lon": 12.5683},
            "Finland": {"code": "FI", "lang": "fi", "capital": "Helsinki", "lat": 60.1699, "lon": 24.9384},
            "Poland": {"code": "PL", "lang": "pl", "capital": "Warsaw", "lat": 52.2297, "lon": 21.0122},
            "Austria": {"code": "AT", "lang": "de", "capital": "Vienna", "lat": 48.2082, "lon": 16.3738},
            "Greece": {"code": "GR", "lang": "el", "capital": "Athens", "lat": 37.9838, "lon": 23.7275},
            "Portugal": {"code": "PT", "lang": "pt", "capital": "Lisbon", "lat": 38.7223, "lon": -9.1393},
            "Ireland": {"code": "IE", "lang": "en", "capital": "Dublin", "lat": 53.3498, "lon": -6.2603},

            # Азия
            "Russia": {"code": "RU", "lang": "ru", "capital": "Moscow", "lat": 55.7558, "lon": 37.6173},
            "China": {"code": "CN", "lang": "zh", "capital": "Beijing", "lat": 39.9042, "lon": 116.4074},
            "Japan": {"code": "JP", "lang": "ja", "capital": "Tokyo", "lat": 35.6762, "lon": 139.6503},
            "South Korea": {"code": "KR", "lang": "ko", "capital": "Seoul", "lat": 37.5665, "lon": 126.9780},
            "India": {"code": "IN", "lang": "en", "capital": "New Delhi", "lat": 28.6139, "lon": 77.2090},
            "Indonesia": {"code": "ID", "lang": "id", "capital": "Jakarta", "lat": -6.2088, "lon": 106.8456},
            "Pakistan": {"code": "PK", "lang": "ur", "capital": "Islamabad", "lat": 33.6844, "lon": 73.0479},
            "Vietnam": {"code": "VN", "lang": "vi", "capital": "Hanoi", "lat": 21.0285, "lon": 105.8542},
            "Thailand": {"code": "TH", "lang": "th", "capital": "Bangkok", "lat": 13.7367, "lon": 100.5231},
            "Malaysia": {"code": "MY", "lang": "ms", "capital": "Kuala Lumpur", "lat": 3.1390, "lon": 101.6869},
            "Singapore": {"code": "SG", "lang": "en", "capital": "Singapore", "lat": 1.3521, "lon": 103.8198},
            "Philippines": {"code": "PH", "lang": "tl", "capital": "Manila", "lat": 14.5995, "lon": 120.9842},

            # Ближний Восток
            "Iran": {"code": "IR", "lang": "fa", "capital": "Tehran", "lat": 35.6892, "lon": 51.3890},
            "Turkey": {"code": "TR", "lang": "tr", "capital": "Ankara", "lat": 39.9334, "lon": 32.8597},
            "Saudi Arabia": {"code": "SA", "lang": "ar", "capital": "Riyadh", "lat": 24.7136, "lon": 46.6753},
            "Israel": {"code": "IL", "lang": "he", "capital": "Jerusalem", "lat": 31.7683, "lon": 35.2137},
            "Egypt": {"code": "EG", "lang": "ar", "capital": "Cairo", "lat": 30.0444, "lon": 31.2357},
            "United Arab Emirates": {"code": "AE", "lang": "ar", "capital": "Abu Dhabi", "lat": 24.4539, "lon": 54.3773},

            # Африка
            "South Africa": {"code": "ZA", "lang": "en", "capital": "Pretoria", "lat": -25.7479, "lon": 28.2293},
            "Nigeria": {"code": "NG", "lang": "en", "capital": "Abuja", "lat": 9.0765, "lon": 7.3986},
            "Kenya": {"code": "KE", "lang": "sw", "capital": "Nairobi", "lat": -1.2921, "lon": 36.8219},

            # Океания
            "Australia": {"code": "AU", "lang": "en", "capital": "Canberra", "lat": -35.2809, "lon": 149.1300},
            "New Zealand": {"code": "NZ", "lang": "en", "capital": "Wellington", "lat": -41.2866, "lon": 174.7756}
        }

        print(f"✅ Инициализировано {len(self.countries_data)} стран")

    def _make_request(self, url: str, params: Dict, retries: int = 3) -> Optional[Dict]:
        """Выполнение HTTP запроса с повторными попытками"""
        for attempt in range(retries):
            try:
                response = requests.get(url, params=params, timeout=10)
                if response.status_code == 200:
                    return response.json()
                elif response.status_code == 429:
                    time.sleep(2 ** attempt)
                else:
                    return None
            except Exception as e:
                print(f"  ⚠️ Ошибка запроса (попытка {attempt + 1}): {e}")
                time.sleep(2)
        return None

    def get_historical_news(self, country: str, topic: str, days_back: int = 30) -> List[Dict]:
        """Получение исторических новостей через NewsAPI"""
        cache_key = f"news_{country}_{topic}_{days_back}"
        if cache_key in self.cache:
            return self.cache[cache_key]

        if country not in self.countries_data:
            return []

        country_data = self.countries_data[country]
        params = {
            "q": f"{topic} {country}",
            "from": (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d"),
            "to": datetime.now().strftime("%Y-%m-%d"),
            "language": country_data["lang"],
            "apiKey": self.news_api_key,
            "pageSize": 50,
            "sortBy": "relevancy"
        }

        print(f"  🌐 Запрос NewsAPI: {topic} в {country}")
        response = self._make_request(f"{self.news_api_url}/everything", params)

        if response and response.get("status") == "ok":
            articles = response.get("articles", [])
            print(f"  ✅ Получено {len(articles)} статей")
            self.cache[cache_key] = articles
            return articles
        return []

    def get_holidays(self, country: str, date: str) -> Dict[str, List[str]]:
        """Получение праздников через Calendarific API"""
        cache_key = f"holidays_{country}_{date}"
        if cache_key in self.cache:
            return self.cache[cache_key]

        if country not in self.countries_data:
            return {"global": [], "national": []}

        country_data = self.countries_data[country]
        year, month, day = date.split('-')
        result = {"global": [], "national": []}

        params = {
            "api_key": self.calendar_api_key,
            "country": country_data["code"],
            "year": year,
            "month": month,
            "day": day
        }

        print(f"  🌐 Запрос Calendarific: праздники в {country}")
        response = self._make_request(f"{self.calendar_api_url}/holidays", params)

        if response and response.get("meta", {}).get("code") == 200:
            holidays = response.get("response", {}).get("holidays", [])
            result["national"] = [h.get("name") for h in holidays if h.get("name")]
            print(f"  ✅ Найдено {len(result['national'])} национальных праздников")

        params["country"] = "global"
        response = self._make_request(f"{self.calendar_api_url}/holidays", params)

        if response and response.get("meta", {}).get("code") == 200:
            holidays = response.get("response", {}).get("holidays", [])
            result["global"] = [h.get("name") for h in holidays if h.get("name")]
            print(f"  ✅ Найдено {len(result['global'])} глобальных праздников")

        self.cache[cache_key] = result
        return result

    def get_weather(self, country: str, date: str) -> Dict:
        """Получение прогноза погоды через OpenWeatherMap API"""
        cache_key = f"weather_{country}_{date}"
        if cache_key in self.cache:
            return self.cache[cache_key]

        if country not in self.countries_data:
            return {}

        country_data = self.countries_data[country]
        params = {
            "lat": country_data["lat"],
            "lon": country_data["lon"],
            "appid": self.weather_api_key,
            "units": "metric",
            "lang": "ru"
        }

        print(f"  🌐 Запрос OpenWeatherMap: погода в {country_data['capital']}")
        response = self._make_request(f"{self.weather_api_url}/forecast", params)

        if response and response.get("cod") == "200":
            target_date = date.split('T')[0] if 'T' in date else date
            for item in response.get("list", []):
                item_date = item.get("dt_txt", "").split(' ')[0]
                if item_date == target_date:
                    result = {
                        "temperature": item.get("main", {}).get("temp"),
                        "feels_like": item.get("main", {}).get("feels_like"),
                        "humidity": item.get("main", {}).get("humidity"),
                        "description": item.get("weather", [{}])[0].get("description"),
                        "wind_speed": item.get("wind", {}).get("speed"),
                        "city": country_data["capital"]
                    }
                    print(f"  ✅ Погода: {result['temperature']}°C, {result['description']}")
                    self.cache[cache_key] = result
                    return result

        return {}

print("✅ Базовый API класс готов")

✅ Базовый API класс готов


In [4]:
# @title
# ============================================
# БЛОК 3: КЛАСС ДЛЯ ПРЕДСКАЗАНИЯ ИЗДАНИЙ
# ============================================

class PublicationPredictor:
    """
    Класс для анализа и предсказания изданий, которые напишут статью
    """

    def __init__(self):
        # Профили изданий по странам
        self.publication_profiles = {
            # Великобритания
            "UK": {
                "The Guardian": {
                    "orientation": "left-leaning",
                    "specialization": ["politics", "environment", "social issues", "healthcare", "autism"],
                    "tone": "progressive",
                    "likelihood_for_autism": 95,
                    "reason": "Активно освещает социальные вопросы, инклюзивность, NHS, благотворительность. "
                              "Имеет постоянную рубрику о нейроразнообразии."
                },
                "BBC News": {
                    "orientation": "neutral",
                    "specialization": ["general news", "health", "politics", "royal family"],
                    "tone": "balanced",
                    "likelihood_for_autism": 92,
                    "reason": "Общественная служба, обязана освещать официальные дни ООН и социальные инициативы. "
                              "Имеет широкую аудиторию и высокий охват."
                },
                "The Times": {
                    "orientation": "center-right",
                    "specialization": ["politics", "business", "health", "education"],
                    "tone": "traditional",
                    "likelihood_for_autism": 75,
                    "reason": "Освещает вопросы здравоохранения с фокусом на политику и реформы NHS."
                },
                "The Telegraph": {
                    "orientation": "right-leaning",
                    "specialization": ["politics", "healthcare", "education"],
                    "tone": "conservative",
                    "likelihood_for_autism": 70,
                    "reason": "Фокус на правительственной политике в области здравоохранения."
                },
                "The Independent": {
                    "orientation": "center-left",
                    "specialization": ["social issues", "health", "human rights"],
                    "tone": "progressive",
                    "likelihood_for_autism": 88,
                    "reason": "Активно освещает права человека, инклюзивность, социальную справедливость."
                }
            },

            # Иран
            "Iran": {
                "Tehran Times": {
                    "orientation": "government-aligned",
                    "specialization": ["politics", "international relations", "culture"],
                    "tone": "official",
                    "likelihood_for_republic_day": 98,
                    "reason": "Официальное англоязычное издание, обязательно освещает государственные праздники."
                },
                "IRNA": {
                    "orientation": "official",
                    "specialization": ["news", "politics", "economy"],
                    "tone": "official",
                    "likelihood_for_republic_day": 100,
                    "reason": "Официальное новостное агентство Ирана, публикует все официальные заявления."
                },
                "Fars News": {
                    "orientation": "conservative",
                    "specialization": ["politics", "military", "culture"],
                    "tone": "patriotic",
                    "likelihood_for_republic_day": 95,
                    "reason": "Освещает патриотические мероприятия и военные парады."
                },
                "Press TV": {
                    "orientation": "government-aligned",
                    "specialization": ["international news", "politics"],
                    "tone": "official",
                    "likelihood_for_republic_day": 90,
                    "reason": "Международная служба, освещает значимые события для внешней аудитории."
                },
                "Mehr News": {
                    "orientation": "moderate",
                    "specialization": ["culture", "politics", "economy"],
                    "tone": "balanced",
                    "likelihood_for_republic_day": 85,
                    "reason": "Освещает культурные и официальные мероприятия."
                }
            },

            # Аргентина
            "Argentina": {
                "Clarín": {
                    "orientation": "center-right",
                    "specialization": ["politics", "national news", "society"],
                    "tone": "traditional",
                    "likelihood_for_malvinas": 95,
                    "reason": "Крупнейшая газета страны, обязательно освещает национальные мемориальные дни."
                },
                "La Nación": {
                    "orientation": "conservative",
                    "specialization": ["politics", "economy", "international"],
                    "tone": "formal",
                    "likelihood_for_malvinas": 93,
                    "reason": "Влиятельное издание с традицией освещения патриотических дат."
                },
                "Página/12": {
                    "orientation": "left-leaning",
                    "specialization": ["politics", "human rights", "society"],
                    "tone": "critical",
                    "likelihood_for_malvinas": 88,
                    "reason": "Активно освещает вопросы суверенитета и национальной идентичности."
                },
                "Infobae": {
                    "orientation": "digital-first",
                    "specialization": ["breaking news", "politics", "society"],
                    "tone": "modern",
                    "likelihood_for_malvinas": 90,
                    "reason": "Цифровое издание с быстрой реакцией на события дня."
                },
                "Ámbito Financiero": {
                    "orientation": "economic",
                    "specialization": ["economy", "business", "politics"],
                    "tone": "analytical",
                    "likelihood_for_malvinas": 75,
                    "reason": "Фокус на экономических аспектах, включая влияние на торговлю."
                }
            },

            # США
            "USA": {
                "The New York Times": {
                    "orientation": "liberal",
                    "specialization": ["national news", "culture", "education"],
                    "tone": "analytical",
                    "likelihood_for_childrens_book": 85,
                    "reason": "Имеет отдел культуры и образования, освещает литературные события."
                },
                "The Washington Post": {
                    "orientation": "liberal",
                    "specialization": ["politics", "education", "culture"],
                    "tone": "investigative",
                    "likelihood_for_childrens_book": 80,
                    "reason": "Освещает образовательные инициативы и культурные события."
                },
                "USA Today": {
                    "orientation": "centrist",
                    "specialization": ["general news", "lifestyle", "education"],
                    "tone": "accessible",
                    "likelihood_for_childrens_book": 75,
                    "reason": "Охватывает широкую аудиторию, включая семейные темы."
                },
                "NPR": {
                    "orientation": "liberal",
                    "specialization": ["education", "culture", "social issues"],
                    "tone": "in-depth",
                    "likelihood_for_childrens_book": 88,
                    "reason": "Общественное радио с фокусом на образование и культуру."
                },
                "PBS News": {
                    "orientation": "neutral",
                    "specialization": ["education", "culture", "public affairs"],
                    "tone": "educational",
                    "likelihood_for_childrens_book": 90,
                    "reason": "Общественное телевидение с программами о грамотности и образовании."
                }
            },

            # Япония
            "Japan": {
                "Nikkei": {
                    "orientation": "business-focused",
                    "specialization": ["economy", "business", "finance"],
                    "tone": "professional",
                    "likelihood_for_business_sakura": 95,
                    "reason": "Главное деловое издание Японии, обязательный обзор начала фискального года."
                },
                "朝日新聞 (Asahi Shimbun)": {
                    "orientation": "liberal",
                    "specialization": ["general news", "business", "culture"],
                    "tone": "balanced",
                    "likelihood_for_business_sakura": 88,
                    "reason": "Крупнейшая газета, освещает и бизнес, и культурные события."
                },
                "読売新聞 (Yomiuri Shimbun)": {
                    "orientation": "conservative",
                    "specialization": ["general news", "business", "sports"],
                    "tone": "traditional",
                    "likelihood_for_business_sakura": 85,
                    "reason": "Самая тиражная газета, освещает все значимые события."
                },
                "日本経済新聞 (Nikkei)": {
                    "orientation": "business",
                    "specialization": ["economy", "finance", "markets"],
                    "tone": "analytical",
                    "likelihood_for_business_sakura": 98,
                    "reason": "Специализированное деловое издание, детальный анализ начала фискального года."
                },
                "NHK": {
                    "orientation": "public",
                    "specialization": ["general news", "business", "culture"],
                    "tone": "official",
                    "likelihood_for_business_sakura": 92,
                    "reason": "Общественное вещание, обязательное освещение государственных и бизнес-событий."
                }
            }
        }

        # Флаги стран
        self.country_flags = {
            "UK": "🇬🇧", "USA": "🇺🇸", "Argentina": "🇦🇷", "Iran": "🇮🇷", "Japan": "🇯🇵",
            "France": "🇫🇷", "Germany": "🇩🇪", "Russia": "🇷🇺", "China": "🇨🇳", "Brazil": "🇧🇷"
        }

    def predict_publications(self, country: str, event_type: str, num_predictions: int = 4) -> List[Dict]:
        """
        Предсказание изданий, которые напишут статью о событии

        Args:
            country: Страна
            event_type: Тип события (autism, republic_day, malvinas_day, etc.)
            num_predictions: Количество изданий для предсказания

        Returns:
            Список изданий с обоснованием
        """
        if country not in self.publication_profiles:
            return self._get_fallback_publications(country, num_predictions)

        profiles = self.publication_profiles[country]

        # Определяем ключ для вероятности
        likelihood_key = f"likelihood_for_{event_type}"

        # Сортируем издания по вероятности
        sorted_publications = sorted(
            profiles.items(),
            key=lambda x: x[1].get(likelihood_key, 50),
            reverse=True
        )

        predictions = []
        for pub_name, pub_data in sorted_publications[:num_predictions]:
            predictions.append({
                "name": pub_name,
                "orientation": pub_data.get("orientation", "unknown"),
                "specialization": pub_data.get("specialization", []),
                "tone": pub_data.get("tone", "balanced"),
                "likelihood": pub_data.get(likelihood_key, 70),
                "reason": pub_data.get("reason", "Имеет историю освещения подобных событий"),
                "flag": self.country_flags.get(country, "🌍")
            })

        return predictions

    def _get_fallback_publications(self, country: str, num_predictions: int) -> List[Dict]:
        """Fallback издания для стран без профиля"""
        fallback = [
            {"name": "National Daily", "orientation": "centrist", "likelihood": 70,
             "reason": "Основное национальное издание"},
            {"name": "The Chronicle", "orientation": "centrist", "likelihood": 65,
             "reason": "Широкий охват аудитории"},
            {"name": "Metro News", "orientation": "centrist", "likelihood": 60,
             "reason": "Цифровое издание с быстрой реакцией"},
            {"name": "Global Times", "orientation": "centrist", "likelihood": 55,
             "reason": "Международная перспектива"}
        ]
        return fallback[:num_predictions]

print("✅ PublicationPredictor готов")

✅ PublicationPredictor готов


In [5]:
# @title
# ============================================
# БЛОК 4: МНОГОЯЗЫЧНЫЙ КЛАСС С ПЕРЕВОДАМИ
# ============================================

class MultiLanguageHandler:
    """
    Класс для работы с переводами
    """

    def __init__(self):
        self.translator = Translator()
        self.language_names = {
            "en": "English", "es": "Spanish", "fr": "French", "de": "German",
            "ru": "Russian", "ja": "Japanese", "zh": "Chinese", "pt": "Portuguese",
            "it": "Italian", "ar": "Arabic", "fa": "Persian (Farsi)", "tr": "Turkish",
            "ko": "Korean", "nl": "Dutch", "sv": "Swedish", "pl": "Polish",
            "el": "Greek", "he": "Hebrew", "th": "Thai", "vi": "Vietnamese"
        }

    def translate_to_english(self, text: str, source_lang: str) -> str:
        """Перевод текста на английский язык"""
        if source_lang == "en" or not text:
            return text

        try:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            translated = loop.run_until_complete(
                self.translator.translate(text, src=source_lang, dest='en')
            )
            loop.close()
            return translated.text
        except Exception as e:
            print(f"  ⚠️ Ошибка перевода: {e}")
            return f"[Translation unavailable] {text}"

    def get_language_name(self, lang_code: str) -> str:
        return self.language_names.get(lang_code, lang_code)

print("✅ MultiLanguageHandler готов")

✅ MultiLanguageHandler готов


In [7]:
# @title
# ============================================
# БЛОК 5: ОПТИМИЗИРОВАННЫЙ КЛАСС С 15-20 ИЗДАНИЯМИ НА СТРАНУ
# ============================================

class OptimizedPublicationPredictor:
    """
    Оптимизированный класс с отфильтрованными самыми релевантными изданиями
    США: 15 изданий, Великобритания: 20 изданий, остальные страны: 15-20
    """

    def __init__(self):
        # Оптимизированные профили изданий (только самые релевантные)
        self.publication_profiles = {

            # ==================== ВЕЛИКОБРИТАНИЯ (20 изданий) ====================
            "UK": {
                # Специализированные медицинские (МАКСИМАЛЬНАЯ ТОЧНОСТЬ) - 5 изданий
                "The Lancet Psychiatry": {
                    "type": "medical_specialist",
                    "specialization": ["psychiatry", "autism", "mental_health"],
                    "audience": "medical_professionals",
                    "likelihood_for_autism": 99,
                    "style": "academic",
                    "reason": "Ведущий мировой журнал по психиатрии - обязательная публикация к World Autism Day"
                },
                "The BMJ (British Medical Journal)": {
                    "type": "medical_specialist",
                    "specialization": ["healthcare", "medical_research", "autism"],
                    "audience": "medical_professionals",
                    "likelihood_for_autism": 98,
                    "style": "academic",
                    "reason": "Один из старейших медицинских журналов, публикует исследования по аутизму"
                },
                "National Autistic Society News": {
                    "type": "charity_specialist",
                    "specialization": ["autism", "neurodiversity", "advocacy"],
                    "audience": "autism_community",
                    "likelihood_for_autism": 100,
                    "style": "advocacy",
                    "reason": "Специализированная организация по аутизму - ГЛАВНЫЙ ИСТОЧНИК"
                },
                "NHS England News": {
                    "type": "official_health",
                    "specialization": ["healthcare", "NHS", "autism_services"],
                    "audience": "public",
                    "likelihood_for_autism": 100,
                    "style": "official",
                    "reason": "Официальный источник NHS - обязательно выпустит заявление"
                },
                "Autism Eye": {
                    "type": "specialist_magazine",
                    "specialization": ["autism", "education", "support"],
                    "audience": "parents_professionals",
                    "likelihood_for_autism": 100,
                    "style": "specialist",
                    "reason": "Специализированный журнал об аутизме - 100% точность"
                },

                # Мейнстримные общенациональные - 6 изданий
                "The Guardian": {
                    "type": "national_newspaper",
                    "specialization": ["social_issues", "healthcare", "autism"],
                    "audience": "general",
                    "likelihood_for_autism": 92,
                    "style": "progressive",
                    "reason": "Активно освещает социальные вопросы, инклюзивность, NHS"
                },
                "BBC News": {
                    "type": "national_broadcaster",
                    "specialization": ["general", "health", "politics"],
                    "audience": "massive",
                    "likelihood_for_autism": 90,
                    "style": "balanced",
                    "reason": "Общественная служба, обязана освещать официальные дни ООН"
                },
                "The Independent": {
                    "type": "national_newspaper",
                    "specialization": ["human_rights", "health", "social_justice"],
                    "audience": "general",
                    "likelihood_for_autism": 88,
                    "style": "progressive",
                    "reason": "Акцент на правах человека и инклюзивности"
                },
                "The Times": {
                    "type": "national_newspaper",
                    "specialization": ["politics", "business", "health"],
                    "audience": "general",
                    "likelihood_for_autism": 75,
                    "style": "traditional",
                    "reason": "Освещает политику в области здравоохранения"
                },
                "The Telegraph": {
                    "type": "national_newspaper",
                    "specialization": ["politics", "healthcare", "education"],
                    "audience": "general",
                    "likelihood_for_autism": 70,
                    "style": "conservative",
                    "reason": "Фокус на правительственной политике"
                },
                "Financial Times": {
                    "type": "business",
                    "specialization": ["economy", "business", "finance"],
                    "audience": "business",
                    "likelihood_for_autism": 40,
                    "style": "analytical",
                    "reason": "Может освещать экономические аспекты поддержки аутизма"
                },

                # Специализированные медицинские и образовательные - 4 издания
                "Health Service Journal": {
                    "type": "healthcare_trade",
                    "specialization": ["NHS", "health_policy", "autism_services"],
                    "audience": "healthcare_professionals",
                    "likelihood_for_autism": 90,
                    "style": "professional",
                    "reason": "Профессиональное издание о здравоохранении"
                },
                "TES (Times Educational Supplement)": {
                    "type": "education_specialist",
                    "specialization": ["education", "SEND", "autism_in_schools"],
                    "audience": "educators",
                    "likelihood_for_autism": 92,
                    "style": "professional",
                    "reason": "Специализированное издание для образования, тема SEND"
                },
                "Schools Week": {
                    "type": "education_trade",
                    "specialization": ["education", "SEND", "policy"],
                    "audience": "educators",
                    "likelihood_for_autism": 88,
                    "style": "professional",
                    "reason": "Образовательная политика и поддержка"
                },
                "The Conversation UK": {
                    "type": "academic",
                    "specialization": ["research", "health", "autism_research"],
                    "audience": "academic",
                    "likelihood_for_autism": 85,
                    "style": "academic",
                    "reason": "Академическая платформа с экспертными мнениями"
                },

                # Региональные (крупнейшие) - 5 изданий
                "Manchester Evening News": {
                    "type": "regional",
                    "specialization": ["local_news", "community", "local_health"],
                    "audience": "regional",
                    "likelihood_for_autism": 75,
                    "style": "community",
                    "reason": "Освещение местных инициатив по аутизму"
                },
                "Birmingham Mail": {
                    "type": "regional",
                    "specialization": ["local_news", "community"],
                    "audience": "regional",
                    "likelihood_for_autism": 72,
                    "style": "community",
                    "reason": "Региональное освещение"
                },
                "Liverpool Echo": {
                    "type": "regional",
                    "specialization": ["local_news", "community"],
                    "audience": "regional",
                    "likelihood_for_autism": 70,
                    "style": "community",
                    "reason": "Северо-запад Англии"
                },
                "The Scotsman": {
                    "type": "regional_national",
                    "specialization": ["scottish_news", "politics", "health"],
                    "audience": "regional",
                    "likelihood_for_autism": 78,
                    "style": "balanced",
                    "reason": "Шотландское издание"
                },
                "Wales Online": {
                    "type": "regional",
                    "specialization": ["welsh_news", "community"],
                    "audience": "regional",
                    "likelihood_for_autism": 68,
                    "style": "local",
                    "reason": "Новости Уэльса"
                }
            },

            # ==================== США (15 изданий) ====================
            "USA": {
                # Специализированные образовательные и библиотечные (МАКСИМАЛЬНАЯ ТОЧНОСТЬ) - 5 изданий
                "School Library Journal": {
                    "type": "library_specialist",
                    "specialization": ["libraries", "childrens_books", "literacy"],
                    "audience": "librarians",
                    "likelihood_for_childrens_book_day": 100,
                    "style": "professional",
                    "reason": "Ведущий журнал для школьных библиотекарей - 100% освещение"
                },
                "Publishers Weekly": {
                    "type": "publishing_trade",
                    "specialization": ["publishing", "childrens_books", "book_industry"],
                    "audience": "publishers",
                    "likelihood_for_childrens_book_day": 99,
                    "style": "trade",
                    "reason": "Главное издание книжной индустрии"
                },
                "The Horn Book": {
                    "type": "childrens_literature",
                    "specialization": ["childrens_books", "reviews", "literature"],
                    "audience": "educators",
                    "likelihood_for_childrens_book_day": 100,
                    "style": "literary",
                    "reason": "Специализированный журнал о детской литературе"
                },
                "Education Week": {
                    "type": "education_specialist",
                    "specialization": ["education", "literacy", "schools"],
                    "audience": "educators",
                    "likelihood_for_childrens_book_day": 98,
                    "style": "professional",
                    "reason": "Ведущее издание об образовании"
                },
                "American Libraries Magazine": {
                    "type": "library_association",
                    "specialization": ["libraries", "literacy", "reading"],
                    "audience": "librarians",
                    "likelihood_for_childrens_book_day": 98,
                    "style": "professional",
                    "reason": "Официальный журнал Американской библиотечной ассоциации"
                },

                # Мейнстримные общенациональные - 6 изданий
                "The New York Times": {
                    "type": "national_newspaper",
                    "specialization": ["national_news", "culture", "education"],
                    "audience": "general",
                    "likelihood_for_childrens_book_day": 85,
                    "style": "analytical",
                    "reason": "Имеет отдел культуры и образования"
                },
                "The Washington Post": {
                    "type": "national_newspaper",
                    "specialization": ["politics", "education", "culture"],
                    "audience": "general",
                    "likelihood_for_childrens_book_day": 80,
                    "style": "investigative",
                    "reason": "Освещает образовательные инициативы"
                },
                "USA Today": {
                    "type": "national_newspaper",
                    "specialization": ["general_news", "lifestyle", "education"],
                    "audience": "general",
                    "likelihood_for_childrens_book_day": 75,
                    "style": "accessible",
                    "reason": "Широкий охват аудитории"
                },
                "NPR": {
                    "type": "public_broadcaster",
                    "specialization": ["education", "culture", "social_issues"],
                    "audience": "general",
                    "likelihood_for_childrens_book_day": 88,
                    "style": "balanced",
                    "reason": "Общественное радио с фокусом на образование"
                },
                "PBS News": {
                    "type": "public_broadcaster",
                    "specialization": ["education", "culture", "public_affairs"],
                    "audience": "general",
                    "likelihood_for_childrens_book_day": 90,
                    "style": "educational",
                    "reason": "Общественное телевидение"
                },
                "CNN": {
                    "type": "cable_news",
                    "specialization": ["breaking_news", "education", "culture"],
                    "audience": "general",
                    "likelihood_for_childrens_book_day": 70,
                    "style": "dynamic",
                    "reason": "Освещение культурных событий"
                },

                # Региональные (крупнейшие) - 4 издания
                "Chicago Tribune": {
                    "type": "regional",
                    "specialization": ["local_news", "education", "culture"],
                    "audience": "regional",
                    "likelihood_for_childrens_book_day": 75,
                    "style": "local",
                    "reason": "Средний Запад США"
                },
                "Los Angeles Times": {
                    "type": "regional",
                    "specialization": ["local_news", "education", "culture"],
                    "audience": "regional",
                    "likelihood_for_childrens_book_day": 78,
                    "style": "local",
                    "reason": "Южная Калифорния"
                },
                "Boston Globe": {
                    "type": "regional",
                    "specialization": ["local_news", "education", "culture"],
                    "audience": "regional",
                    "likelihood_for_childrens_book_day": 72,
                    "style": "local",
                    "reason": "Новая Англия"
                },
                "The Seattle Times": {
                    "type": "regional",
                    "specialization": ["local_news", "education", "culture"],
                    "audience": "regional",
                    "likelihood_for_childrens_book_day": 70,
                    "style": "local",
                    "reason": "Тихоокеанский Северо-Запад"
                }
            },

            # ==================== ИРАН (15 изданий) ====================
            "Iran": {
                # Официальные государственные - 5 изданий
                "IRNA (Islamic Republic News Agency)": {
                    "type": "official_agency",
                    "specialization": ["official_news", "politics", "republic_day"],
                    "audience": "national",
                    "likelihood_for_republic_day": 100,
                    "style": "official",
                    "reason": "Официальное государственное агентство - 100% освещение"
                },
                "Fars News Agency": {
                    "type": "conservative",
                    "specialization": ["politics", "military", "revolution"],
                    "audience": "national",
                    "likelihood_for_republic_day": 100,
                    "style": "patriotic",
                    "reason": "Освещение военных парадов и официальных церемоний"
                },
                "Mehr News Agency": {
                    "type": "semi_official",
                    "specialization": ["general_news", "culture", "politics"],
                    "audience": "national",
                    "likelihood_for_republic_day": 98,
                    "style": "balanced",
                    "reason": "Широкое освещение официальных мероприятий"
                },
                "Tasnim News Agency": {
                    "type": "conservative",
                    "specialization": ["politics", "military", "revolution"],
                    "audience": "national",
                    "likelihood_for_republic_day": 98,
                    "style": "patriotic",
                    "reason": "Связана с Корпусом стражей исламской революции"
                },

                # Международные - 4 издания
                "Press TV": {
                    "type": "international",
                    "specialization": ["international_news", "politics"],
                    "audience": "international",
                    "likelihood_for_republic_day": 95,
                    "style": "official",
                    "reason": "Международная служба для внешней аудитории"
                },
                "Tehran Times": {
                    "type": "english_language",
                    "specialization": ["international_news", "politics", "culture"],
                    "audience": "international",
                    "likelihood_for_republic_day": 96,
                    "style": "official",
                    "reason": "Англоязычное издание"
                },
                "Iran Daily": {
                    "type": "english_language",
                    "specialization": ["daily_news", "politics"],
                    "audience": "international",
                    "likelihood_for_republic_day": 94,
                    "style": "official",
                    "reason": "Ежедневная англоязычная газета"
                },

                # Реформистские - 3 издания
                "Shargh Daily": {
                    "type": "reformist",
                    "specialization": ["politics", "society", "culture"],
                    "audience": "national",
                    "likelihood_for_republic_day": 85,
                    "style": "critical",
                    "reason": "Реформистское издание"
                },
                "Etemad": {
                    "type": "reformist",
                    "specialization": ["politics", "society"],
                    "audience": "national",
                    "likelihood_for_republic_day": 82,
                    "style": "moderate",
                    "reason": "Реформистская газета"
                },
                "Arman Daily": {
                    "type": "reformist",
                    "specialization": ["youth", "society", "culture"],
                    "audience": "youth",
                    "likelihood_for_republic_day": 78,
                    "style": "modern",
                    "reason": "Ориентирована на молодежь"
                },

                # Государственное телевидение - 3 издания
                "IRIB (Islamic Republic of Iran Broadcasting)": {
                    "type": "state_broadcaster",
                    "specialization": ["tv", "radio", "official_news"],
                    "audience": "massive",
                    "likelihood_for_republic_day": 100,
                    "style": "official",
                    "reason": "Государственное телевидение - прямая трансляция"
                },
                "IRIB News": {
                    "type": "state_broadcaster",
                    "specialization": ["tv_news", "official_news"],
                    "audience": "national",
                    "likelihood_for_republic_day": 99,
                    "style": "official",
                    "reason": "Новостная служба государственного ТВ"
                },
                "IRNA English": {
                    "type": "official_english",
                    "specialization": ["international", "official_news"],
                    "audience": "international",
                    "likelihood_for_republic_day": 98,
                    "style": "official",
                    "reason": "Англоязычная версия официального агентства"
                }
            },

            # ==================== АРГЕНТИНА (15 изданий) ====================
            "Argentina": {
                # Специализированные военные/ветеранские - 4 издания
                "Centro de Ex Combatientes de Malvinas": {
                    "type": "veteran_organization",
                    "specialization": ["veterans", "malvinas", "memorial"],
                    "audience": "veterans",
                    "likelihood_for_malvinas_day": 100,
                    "style": "memorial",
                    "reason": "Официальная организация ветеранов Мальвин - 100% освещение"
                },
                "Soldados Digital": {
                    "type": "military_specialist",
                    "specialization": ["military", "veterans", "malvinas"],
                    "audience": "military",
                    "likelihood_for_malvinas_day": 100,
                    "style": "patriotic",
                    "reason": "Военное издание"
                },
                "Malvinas Argentinas": {
                    "type": "specialist",
                    "specialization": ["malvinas", "sovereignty"],
                    "audience": "niche",
                    "likelihood_for_malvinas_day": 100,
                    "style": "patriotic",
                    "reason": "Специализированное издание по теме Мальвин"
                },
                "El Diario del Fin del Mundo": {
                    "type": "regional",
                    "specialization": ["ushuaia", "malvinas", "local"],
                    "audience": "regional",
                    "likelihood_for_malvinas_day": 98,
                    "style": "local",
                    "reason": "Газета Ушуайи - ключевое место церемоний"
                },

                # Мейнстримные общенациональные - 5 изданий
                "Clarín": {
                    "type": "national_newspaper",
                    "specialization": ["politics", "national_news", "malvinas"],
                    "audience": "massive",
                    "likelihood_for_malvinas_day": 95,
                    "style": "traditional",
                    "reason": "Крупнейшая газета Аргентины"
                },
                "La Nación": {
                    "type": "national_newspaper",
                    "specialization": ["politics", "international", "malvinas"],
                    "audience": "large",
                    "likelihood_for_malvinas_day": 94,
                    "style": "formal",
                    "reason": "Влиятельное консервативное издание"
                },
                "Página/12": {
                    "type": "national_newspaper",
                    "specialization": ["politics", "human_rights", "malvinas"],
                    "audience": "medium",
                    "likelihood_for_malvinas_day": 92,
                    "style": "critical",
                    "reason": "Акцент на правах человека и суверенитете"
                },
                "Infobae": {
                    "type": "digital",
                    "specialization": ["breaking_news", "politics", "malvinas"],
                    "audience": "massive",
                    "likelihood_for_malvinas_day": 94,
                    "style": "modern",
                    "reason": "Цифровое издание с быстрой реакцией"
                },
                "Ámbito Financiero": {
                    "type": "economic",
                    "specialization": ["economy", "business", "politics"],
                    "audience": "business",
                    "likelihood_for_malvinas_day": 70,
                    "style": "analytical",
                    "reason": "Экономический аспект"
                },

                # Региональные - 6 изданий
                "La Opinión Austral": {
                    "type": "regional",
                    "specialization": ["patagonia", "malvinas"],
                    "audience": "regional",
                    "likelihood_for_malvinas_day": 95,
                    "style": "local",
                    "reason": "Региональное издание Патагонии"
                },
                "La Nueva Provincia": {
                    "type": "regional",
                    "specialization": ["bahia_blanca", "local_news"],
                    "audience": "regional",
                    "likelihood_for_malvinas_day": 85,
                    "style": "local",
                    "reason": "Южная провинция Буэнос-Айрес"
                },
                "Los Andes": {
                    "type": "regional",
                    "specialization": ["mendoza", "local_news"],
                    "audience": "regional",
                    "likelihood_for_malvinas_day": 82,
                    "style": "local",
                    "reason": "Провинция Мендоса"
                },
                "La Voz del Interior": {
                    "type": "regional",
                    "specialization": ["cordoba", "local_news"],
                    "audience": "regional",
                    "likelihood_for_malvinas_day": 80,
                    "style": "local",
                    "reason": "Провинция Кордова"
                },
                "El Litoral": {
                    "type": "regional",
                    "specialization": ["santa_fe", "local_news"],
                    "audience": "regional",
                    "likelihood_for_malvinas_day": 78,
                    "style": "local",
                    "reason": "Провинция Санта-Фе"
                },
                "Diario Uno": {
                    "type": "regional",
                    "specialization": ["mendoza", "local_news"],
                    "audience": "regional",
                    "likelihood_for_malvinas_day": 75,
                    "style": "local",
                    "reason": "Сеть региональных изданий"
                }
            },

            # ==================== ЯПОНИЯ (15 изданий) ====================
            "Japan": {
                # Деловые специализированные - 4 издания
                "日本経済新聞 (Nikkei)": {
                    "type": "business_daily",
                    "specialization": ["economy", "business", "finance"],
                    "audience": "business",
                    "likelihood_for_business_sakura": 98,
                    "style": "professional",
                    "reason": "Главная деловая газета Японии"
                },
                "日経ビジネス (Nikkei Business)": {
                    "type": "business_magazine",
                    "specialization": ["business", "corporate_strategy"],
                    "audience": "business",
                    "likelihood_for_business_sakura": 95,
                    "style": "analytical",
                    "reason": "Ведущий деловой журнал"
                },
                "東洋経済 (Toyo Keizai)": {
                    "type": "economic_weekly",
                    "specialization": ["economy", "business", "finance"],
                    "audience": "business",
                    "likelihood_for_business_sakura": 92,
                    "style": "analytical",
                    "reason": "Влиятельный экономический еженедельник"
                },
                "週刊ダイヤモンド (Weekly Diamond)": {
                    "type": "business_weekly",
                    "specialization": ["business", "economy"],
                    "audience": "business",
                    "likelihood_for_business_sakura": 90,
                    "style": "analytical",
                    "reason": "Популярный деловой еженедельник"
                },

                # Туристические (для сезона сакуры) - 3 издания
                "じゃらん (Jalan)": {
                    "type": "travel_magazine",
                    "specialization": ["travel", "sakura", "tourism"],
                    "audience": "travelers",
                    "likelihood_for_business_sakura": 95,
                    "style": "lifestyle",
                    "reason": "Крупнейший туристический журнал"
                },
                "るるぶ (Rurubu)": {
                    "type": "travel_guide",
                    "specialization": ["travel", "sakura", "destinations"],
                    "audience": "travelers",
                    "likelihood_for_business_sakura": 94,
                    "style": "guide",
                    "reason": "Популярный путеводитель"
                },
                "旅の手帖 (Tabi no Techo)": {
                    "type": "travel_magazine",
                    "specialization": ["travel", "culture", "seasonal"],
                    "audience": "travelers",
                    "likelihood_for_business_sakura": 90,
                    "style": "cultural",
                    "reason": "Журнал о путешествиях и культуре"
                },

                # Общенациональные - 5 изданий
                "朝日新聞 (Asahi Shimbun)": {
                    "type": "national_newspaper",
                    "specialization": ["general", "business", "culture"],
                    "audience": "general",
                    "likelihood_for_business_sakura": 85,
                    "style": "balanced",
                    "reason": "Крупнейшая газета"
                },
                "読売新聞 (Yomiuri Shimbun)": {
                    "type": "national_newspaper",
                    "specialization": ["general", "business", "sports"],
                    "audience": "general",
                    "likelihood_for_business_sakura": 82,
                    "style": "traditional",
                    "reason": "Самая тиражная газета"
                },
                "毎日新聞 (Mainichi Shimbun)": {
                    "type": "national_newspaper",
                    "specialization": ["general", "business", "culture"],
                    "audience": "general",
                    "likelihood_for_business_sakura": 80,
                    "style": "balanced",
                    "reason": "Влиятельная национальная газета"
                },
                "NHK": {
                    "type": "public_broadcaster",
                    "specialization": ["general", "business", "culture"],
                    "audience": "massive",
                    "likelihood_for_business_sakura": 92,
                    "style": "official",
                    "reason": "Общественное телевидение"
                },
                "テレビ東京 (TV Tokyo)": {
                    "type": "broadcaster",
                    "specialization": ["business", "economy", "news"],
                    "audience": "general",
                    "likelihood_for_business_sakura": 88,
                    "style": "business_focused",
                    "reason": "Специализация на бизнес-новостях"
                },

                # Региональные - 3 издания
                "東京新聞 (Tokyo Shimbun)": {
                    "type": "regional",
                    "specialization": ["tokyo", "local_news", "sakura"],
                    "audience": "regional",
                    "likelihood_for_business_sakura": 88,
                    "style": "local",
                    "reason": "Новости Токио"
                },
                "京都新聞 (Kyoto Shimbun)": {
                    "type": "regional",
                    "specialization": ["kyoto", "culture", "sakura"],
                    "audience": "regional",
                    "likelihood_for_business_sakura": 85,
                    "style": "cultural",
                    "reason": "Новости Киото - культурной столицы"
                },
                "大阪日日新聞 (Osaka Nichinichi)": {
                    "type": "regional",
                    "specialization": ["osaka", "business", "local"],
                    "audience": "regional",
                    "likelihood_for_business_sakura": 82,
                    "style": "local",
                    "reason": "Новости Осаки"
                }
            }
        }

        self.country_flags = {
            "UK": "🇬🇧", "USA": "🇺🇸", "Argentina": "🇦🇷",
            "Iran": "🇮🇷", "Japan": "🇯🇵"
        }

    def get_top_publications(self, country: str, event_type: str, num: int = 10) -> List[Dict]:
        """Получить топ изданий для события"""
        if country not in self.publication_profiles:
            return self._get_fallback(country, num)

        profiles = self.publication_profiles[country]
        likelihood_key = f"likelihood_for_{event_type}"

        publications = []
        for name, data in profiles.items():
            if likelihood_key in data:
                publications.append({
                    "name": name,
                    "type": data["type"],
                    "audience": data["audience"],
                    "style": data["style"],
                    "likelihood": data[likelihood_key],
                    "reason": data["reason"],
                    "flag": self.country_flags.get(country, "🌍")
                })

        # Сортируем по вероятности
        publications.sort(key=lambda x: x["likelihood"], reverse=True)
        return publications[:num]

    def get_all_publications(self, country: str) -> List[Dict]:
        """Получить все издания для страны"""
        if country not in self.publication_profiles:
            return []

        profiles = self.publication_profiles[country]
        publications = []
        for name, data in profiles.items():
            publications.append({
                "name": name,
                "type": data["type"],
                "audience": data["audience"],
                "style": data["style"],
                "reason": data.get("reason", ""),
                "flag": self.country_flags.get(country, "🌍")
            })
        return publications

    def _get_fallback(self, country: str, num: int) -> List[Dict]:
        """Fallback издания"""
        return [
            {"name": "National Daily", "type": "general", "likelihood": 70, "style": "neutral",
             "reason": "Основное национальное издание", "flag": "🌍"},
            {"name": "The Chronicle", "type": "general", "likelihood": 65, "style": "neutral",
             "reason": "Широкий охват аудитории", "flag": "🌍"}
        ][:num]

print("✅ OptimizedPublicationPredictor готов")
print(f"\n📊 ИТОГОВАЯ СТАТИСТИКА ИЗДАНИЙ:")
print(f"  🇬🇧 Великобритания: 20 изданий")
print(f"  🇺🇸 США: 15 изданий")
print(f"  🇮🇷 Иран: 15 изданий")
print(f"  🇦🇷 Аргентина: 15 изданий")
print(f"  🇯🇵 Япония: 15 изданий")
print(f"  📰 ВСЕГО: 80 изданий")

✅ OptimizedPublicationPredictor готов

📊 ИТОГОВАЯ СТАТИСТИКА ИЗДАНИЙ:
  🇬🇧 Великобритания: 20 изданий
  🇺🇸 США: 15 изданий
  🇮🇷 Иран: 15 изданий
  🇦🇷 Аргентина: 15 изданий
  🇯🇵 Япония: 15 изданий
  📰 ВСЕГО: 80 изданий


In [8]:
# @title
# ============================================
# БЛОК 5.1: РАСЧЕТ API ЗАПРОСОВ ДЛЯ ОПТИМИЗИРОВАННОГО СПИСКА
# ============================================

print("\n" + "="*80)
print("📊 РАСЧЕТ API ЗАПРОСОВ ДЛЯ 80 ОПТИМИЗИРОВАННЫХ ИЗДАНИЙ")
print("="*80)

# Итоговые данные
publications_by_country = {
    "UK": 20,
    "USA": 15,
    "Iran": 15,
    "Argentina": 15,
    "Japan": 15
}

total_publications = sum(publications_by_country.values())

# Расчет запросов
newsapi_requests = total_publications  # по 1 на издание
calendarific_requests = len(publications_by_country)  # 1 на страну (апрель целиком)
weather_requests = len(publications_by_country)  # 1 на страну
wikipedia_requests = 1  # 1 запрос на 2 апреля

total_requests = newsapi_requests + calendarific_requests + weather_requests + wikipedia_requests

print(f"\n📚 РАСПРЕДЕЛЕНИЕ ИЗДАНИЙ:")
for country, count in publications_by_country.items():
    flag = {"UK": "🇬🇧", "USA": "🇺🇸", "Iran": "🇮🇷", "Argentina": "🇦🇷", "Japan": "🇯🇵"}[country]
    print(f"  {flag} {country}: {count} изданий")
print(f"\n  📰 ВСЕГО: {total_publications} изданий")

print(f"\n🔢 API ЗАПРОСЫ:")
print(f"  • NewsAPI: {newsapi_requests} запросов (по 1 на каждое издание)")
print(f"  • Calendarific: {calendarific_requests} запросов (1 на страну за апрель)")
print(f"  • OpenWeatherMap: {weather_requests} запросов (1 на страну)")
print(f"  • Wikipedia: {wikipedia_requests} запросов")
print(f"  {'='*40}")
print(f"  🔢 ИТОГО: {total_requests} API запросов")

print(f"\n📊 ПРОВЕРКА ЛИМИТОВ:")
print(f"  • NewsAPI: {newsapi_requests} / 100 = {'✅ ВПИСЫВАЕТСЯ' if newsapi_requests <= 100 else '❌ ПРЕВЫШАЕТ'}")
print(f"  • Calendarific: {calendarific_requests} / 1000 = ✅ ВПИСЫВАЕТСЯ")
print(f"  • OpenWeatherMap: {weather_requests} / 60 = ✅ ВПИСЫВАЕТСЯ")

print(f"\n{'='*80}")
print("✅ ОПТИМИЗАЦИЯ УСПЕШНА!")
print(f"   Сокращение с 121 до 80 изданий (-34%)")
print(f"   Сокращение API запросов с 132 до {total_requests} (-{100 - int(total_requests/132*100)}%)")
print(f"   Все запросы в пределах бесплатных лимитов!")
print("="*80)


📊 РАСЧЕТ API ЗАПРОСОВ ДЛЯ 80 ОПТИМИЗИРОВАННЫХ ИЗДАНИЙ

📚 РАСПРЕДЕЛЕНИЕ ИЗДАНИЙ:
  🇬🇧 UK: 20 изданий
  🇺🇸 USA: 15 изданий
  🇮🇷 Iran: 15 изданий
  🇦🇷 Argentina: 15 изданий
  🇯🇵 Japan: 15 изданий

  📰 ВСЕГО: 80 изданий

🔢 API ЗАПРОСЫ:
  • NewsAPI: 80 запросов (по 1 на каждое издание)
  • Calendarific: 5 запросов (1 на страну за апрель)
  • OpenWeatherMap: 5 запросов (1 на страну)
  • Wikipedia: 1 запросов
  🔢 ИТОГО: 91 API запросов

📊 ПРОВЕРКА ЛИМИТОВ:
  • NewsAPI: 80 / 100 = ✅ ВПИСЫВАЕТСЯ
  • Calendarific: 5 / 1000 = ✅ ВПИСЫВАЕТСЯ
  • OpenWeatherMap: 5 / 60 = ✅ ВПИСЫВАЕТСЯ

✅ ОПТИМИЗАЦИЯ УСПЕШНА!
   Сокращение с 121 до 80 изданий (-34%)
   Сокращение API запросов с 132 до 91 (-32%)
   Все запросы в пределах бесплатных лимитов!


In [9]:
# @title
# ============================================
# БЛОК 6: ГЕНЕРАЦИЯ ЗАГОЛОВКОВ И ПЕРВЫХ АБЗАЦЕВ
# ============================================

class HeadlineGenerator:
    """
    Класс для генерации заголовков и первых абзацев в стиле конкретных изданий
    """

    def __init__(self, publication_predictor: OptimizedPublicationPredictor):
        self.pub_predictor = publication_predictor

        # Шаблоны заголовков по стилям изданий
        self.headline_templates = {
            "academic": [
                "{event}: New research findings published in {country}",
                "Study reveals {adjective} trends in {event} management",
                "The Lancet Psychiatry: {breakthrough} in {event} understanding",
                "Systematic review highlights {key_finding} in {event} care"
            ],
            "official": [
                "Government announces new {event} initiative for {year}",
                "Official statement on {event} released today",
                "Ministry confirms {event} support programs expansion",
                "Cabinet approves £{amount}M for {event} services"
            ],
            "advocacy": [
                "\"{slogan}\": Campaigners demand {event} reform in {country}",
                "{group} calls for greater {event} support and awareness",
                "New report highlights {challenge} in {event} services",
                "Families urge government to prioritize {event} funding"
            ],
            "specialist": [
                "{community} marks {event}: {outlook} ahead",
                "Special report: State of {event} services in {country}",
                "Expert analysis: The {aspect} of {event} support in {year}",
                "Professional guidelines for {event} management updated"
            ],
            "progressive": [
                "Landmark {event} strategy unveiled by {government}",
                "{event} rights: New era of {change} begins",
                "\"{vision}\": Minister pledges transformative {event} reform",
                "Groundbreaking initiative promises {benefit} for {affected}"
            ],
            "balanced": [
                "{event}: What you need to know today",
                "{country} marks {holiday} with new {event} initiatives",
                "Analysis: The state of {event} in {country}",
                "Everything you need to understand about {event}"
            ],
            "patriotic": [
                "{country} celebrates {holiday} with nationwide ceremonies",
                "President leads {holiday} commemorations across the nation",
                "\"{patriotic_phrase}\": {country} unites for {holiday}",
                "National pride on display as {country} marks {holiday}"
            ],
            "memorial": [
                "{country} remembers: {holiday} commemorations held nationwide",
                "\"{memorial_phrase}\": Veterans gather for {holiday}",
                "Emotional tributes mark {holiday} in {country}",
                "Fallen heroes honored on {holiday}"
            ],
            "professional": [
                "New fiscal year begins: Corporate {event} strategies unveiled",
                "Business outlook: {event} sector shows {trend}",
                "Market analysis: {event} trends for {year}",
                "{company} announces {initiative} for new fiscal year"
            ],
            "trade": [
                "Publishing industry celebrates {event}: New {category} announced",
                "Book trade rallies for {event}: {initiative} launched",
                "Industry leaders gather for {event} celebrations",
                "Publishers unveil {number} new titles for {event}"
            ],
            "community": [
                "Local {event} initiatives receive boost in {city}",
                "Community comes together for {event} in {region}",
                "{city} residents mark {event} with special events",
                "Grassroots {event} programs gain momentum"
            ],
            "cultural": [
                "Cherry blossom season peaks as {event} begins",
                "{tradition} celebrated alongside {event} in {city}",
                "Cultural festivities mark {event} in {region}",
                "{event} brings {cultural_aspect} to forefront"
            ]
        }

        # Шаблоны первых абзацев
        self.paragraph_templates = {
            "academic": [
                "A new study published in {journal} reveals that {finding}. Researchers analyzed {sample} and found {result}, suggesting significant implications for {event} care and management."
            ],
            "official": [
                "The {ministry} has announced a comprehensive {event} strategy today. According to official sources, the £{amount}M initiative aims to {goal} and will be implemented across {country} by {deadline}."
            ],
            "advocacy": [
                "{organization} is calling for urgent {event} reform as new data shows {statistic}. Speaking at a press conference, {spokesperson} said: \"{quote}\", highlighting the growing need for support."
            ],
            "specialist": [
                "As the world marks {event}, specialists in {country} are drawing attention to {issue}. New data from {source} indicates {finding}, prompting calls for {action}."
            ],
            "progressive": [
                "In a significant development, {country} has unveiled what officials call a \"{vision}\" approach to {event}. The strategy, developed with input from {stakeholders}, aims to {goal} by {deadline}."
            ],
            "balanced": [
                "Today marks {event} in {country}, with officials and advocates weighing in on progress and challenges. Recent data shows {statistic}, while experts note {trend} in service provision."
            ],
            "patriotic": [
                "Thousands gathered in {city} today to celebrate {holiday}, with President {president} leading official ceremonies. The event marks {years} years since {historical_event}, a pivotal moment in {country}'s history."
            ],
            "memorial": [
                "Argentina today commemorates the {years}th anniversary of the Malvinas War, honoring the veterans and fallen soldiers who defended national sovereignty. President {president} led the official ceremony in Buenos Aires, laying a wreath at the Monumento a los Caídos."
            ],
            "professional": [
                "As Japan's new fiscal year begins, major corporations are unveiling their strategies for {year}. The Nikkei 225 opened {direction} today, reflecting {sentiment} about the economic outlook despite global uncertainties."
            ],
            "trade": [
                "The publishing industry is celebrating International Children's Book Day with a major initiative: {initiative}. Publishers, libraries and educators are joining forces to promote literacy and a love of reading among young people."
            ]
        }

    def _get_adjective(self) -> str:
        """Случайное прилагательное"""
        adjectives = ["significant", "unprecedented", "notable", "remarkable", "important", "groundbreaking"]
        return random.choice(adjectives)

    def _get_breakthrough(self) -> str:
        """Случайное описание прорыва"""
        breakthroughs = ["breakthrough in early diagnosis", "novel treatment approach",
                         "innovative support model", "comprehensive care framework"]
        return random.choice(breakthroughs)

    def _get_key_finding(self) -> str:
        """Случайное ключевое открытие"""
        findings = ["improved outcomes", "reduced waiting times", "better integration",
                   "enhanced support systems", "increased awareness"]
        return random.choice(findings)

    def _get_slogan(self) -> str:
        """Случайный слоган"""
        slogans = ["We need action now", "Time for change", "Support not stigma",
                   "Nothing about us without us", "Inclusion matters"]
        return random.choice(slogans)

    def _get_challenge(self) -> str:
        """Случайное описание проблемы"""
        challenges = ["growing waiting lists", "workplace discrimination", "education barriers",
                     "healthcare access gaps", "social isolation"]
        return random.choice(challenges)

    def generate_headline(self, publication: Dict, event_type: str, country: str,
                          shared_data: Dict) -> str:
        """
        Генерация заголовка в стиле издания
        """
        style = publication["style"]
        templates = self.headline_templates.get(style, self.headline_templates["balanced"])
        template = random.choice(templates)

        # Подготовка переменных
        variables = {
            "event": self._get_event_name(event_type, country),
            "country": country,
            "year": "2026",
            "amount": random.choice(["75", "100", "125", "150"]),
            "adjective": self._get_adjective(),
            "breakthrough": self._get_breakthrough(),
            "key_finding": self._get_key_finding(),
            "slogan": self._get_slogan(),
            "challenge": self._get_challenge(),
            "holiday": ", ".join(shared_data.get("holidays", [])[:2]) if shared_data.get("holidays") else event_type,
            "government": "the government",
            "change": "positive change",
            "vision": "visionary",
            "benefit": "improved outcomes",
            "affected": "those affected",
            "patriotic_phrase": "Pride and honor",
            "memorial_phrase": "Never forget",
            "trend": "steady growth",
            "initiative": "major investment",
            "category": "children's literature",
            "number": random.choice(["500", "1000", "5000"]),
            "city": self._get_city(country),
            "region": self._get_region(country),
            "tradition": "Hanami",
            "cultural_aspect": "traditional customs"
        }

        # Заменяем переменные
        headline = template
        for key, value in variables.items():
            headline = headline.replace(f"{{{key}}}", str(value))

        return headline

    def generate_first_paragraph(self, publication: Dict, event_type: str, country: str,
                                 shared_data: Dict) -> str:
        """
        Генерация первого абзаца в стиле издания
        """
        style = publication["style"]
        templates = self.paragraph_templates.get(style, self.paragraph_templates["balanced"])
        template = random.choice(templates)

        # Специфичные переменные для разных событий
        event_specific = self._get_event_specific_variables(event_type, country, shared_data)

        # Общие переменные
        variables = {
            "event": self._get_event_name(event_type, country),
            "country": country,
            "journal": publication["name"],
            "finding": event_specific.get("finding", "significant progress"),
            "sample": event_specific.get("sample", "comprehensive data"),
            "result": event_specific.get("result", "positive outcomes"),
            "ministry": self._get_ministry(country),
            "amount": random.choice(["75", "100", "125", "150"]),
            "goal": event_specific.get("goal", "improve services"),
            "deadline": "2028",
            "organization": self._get_organization(event_type, country),
            "statistic": event_specific.get("statistic", "1 in 100 children"),
            "spokesperson": event_specific.get("spokesperson", "the spokesperson"),
            "quote": event_specific.get("quote", "This is a crucial step forward"),
            "issue": event_specific.get("issue", "increasing demand"),
            "source": event_specific.get("source", "official statistics"),
            "action": event_specific.get("action", "immediate action"),
            "vision": event_specific.get("vision", "inclusive future"),
            "stakeholders": event_specific.get("stakeholders", "experts and advocates"),
            "holiday": ", ".join(shared_data.get("holidays", [])[:2]) if shared_data.get("holidays") else event_type,
            "city": self._get_city(country),
            "president": self._get_president(country),
            "years": self._get_years(event_type),
            "historical_event": self._get_historical_event(event_type, country),
            "direction": random.choice(["higher", "lower", "steady"]),
            "sentiment": random.choice(["optimism", "caution", "confidence"]),
            "initiative": event_specific.get("initiative", "a major literacy campaign")
        }

        # Заменяем переменные
        paragraph = template
        for key, value in variables.items():
            paragraph = paragraph.replace(f"{{{key}}}", str(value))

        return paragraph

    def _get_event_name(self, event_type: str, country: str) -> str:
        """Получение названия события"""
        names = {
            "autism": "World Autism Awareness Day",
            "republic_day": "Islamic Republic Day",
            "malvinas_day": "Malvinas Veterans Day",
            "childrens_book_day": "International Children's Book Day",
            "business_sakura": "the new fiscal year and cherry blossom season"
        }
        return names.get(event_type, event_type)

    def _get_city(self, country: str) -> str:
        """Получение столицы или крупного города"""
        cities = {
            "UK": "London",
            "USA": "Washington D.C.",
            "Iran": "Tehran",
            "Argentina": "Buenos Aires",
            "Japan": "Tokyo"
        }
        return cities.get(country, "the capital")

    def _get_region(self, country: str) -> str:
        """Получение региона"""
        regions = {
            "UK": "the United Kingdom",
            "USA": "the United States",
            "Iran": "the nation",
            "Argentina": "the country",
            "Japan": "the archipelago"
        }
        return regions.get(country, "the country")

    def _get_ministry(self, country: str) -> str:
        """Получение названия министерства"""
        ministries = {
            "UK": "Department of Health and Social Care",
            "USA": "Department of Education",
            "Iran": "Ministry of Culture and Islamic Guidance",
            "Argentina": "Ministry of Defense",
            "Japan": "Ministry of Economy, Trade and Industry"
        }
        return ministries.get(country, "the relevant ministry")

    def _get_president(self, country: str) -> str:
        """Получение имени президента/лидера"""
        leaders = {
            "UK": "Sir Keir Starmer",
            "USA": "Donald Trump",
            "Iran": "Masoud Pezeshkian",
            "Argentina": "Javier Milei",
            "Japan": "Shigeru Ishiba"
        }
        return leaders.get(country, "the President")

    def _get_organization(self, event_type: str, country: str) -> str:
        """Получение названия организации"""
        orgs = {
            ("autism", "UK"): "the National Autistic Society",
            ("autism", "USA"): "Autism Speaks",
            ("republic_day", "Iran"): "the Islamic Republic News Agency",
            ("malvinas_day", "Argentina"): "the Malvinas Veterans Association",
            ("childrens_book_day", "USA"): "the American Library Association",
            ("business_sakura", "Japan"): "the Japan Business Federation"
        }
        return orgs.get((event_type, country), "leading organizations")

    def _get_years(self, event_type: str) -> str:
        """Получение количества лет"""
        years = {
            "autism": "19",
            "republic_day": "47",
            "malvinas_day": "44",
            "childrens_book_day": "60",
            "business_sakura": "2026"
        }
        return years.get(event_type, "many")

    def _get_historical_event(self, event_type: str, country: str) -> str:
        """Получение исторического события"""
        events = {
            ("republic_day", "Iran"): "the establishment of the Islamic Republic in 1979",
            ("malvinas_day", "Argentina"): "the Malvinas War in 1982",
            ("autism", "UK"): "the first Autism Awareness Day in 2007"
        }
        return events.get((event_type, country), "this historic occasion")

    def _get_event_specific_variables(self, event_type: str, country: str,
                                      shared_data: Dict) -> Dict:
        """Получение специфичных для события переменных"""
        if event_type == "autism":
            return {
                "finding": "significant improvements in early diagnosis",
                "sample": "data from 50,000 children",
                "result": "a 30% reduction in waiting times",
                "goal": "reduce diagnosis waiting times by 50%",
                "statistic": "1 in 100 children diagnosed with autism",
                "spokesperson": "the NAS chief executive",
                "quote": "This investment will transform lives",
                "issue": "growing demand for assessments",
                "source": "NHS England",
                "action": "urgent reform",
                "vision": "inclusive and accessible services",
                "stakeholders": "autistic people, families, and healthcare professionals",
                "initiative": "a comprehensive autism strategy"
            }
        elif event_type == "republic_day":
            return {
                "finding": "remarkable progress since 1979",
                "result": "continued resilience and development",
                "goal": "strengthen national unity",
                "statistic": "47 years of achievements",
                "quote": "Our nation stands proud and strong",
                "issue": "national sovereignty",
                "source": "official government reports",
                "action": "celebrating our heritage",
                "vision": "prosperous future",
                "stakeholders": "government officials and citizens",
                "initiative": "national development programs"
            }
        elif event_type == "malvinas_day":
            return {
                "finding": "enduring commitment to sovereignty",
                "result": "continued international support",
                "goal": "honor the fallen heroes",
                "statistic": "649 fallen soldiers",
                "quote": "Malvinas is Argentine soil",
                "issue": "sovereignty claim",
                "source": "historical records",
                "action": "diplomatic efforts",
                "vision": "peaceful resolution",
                "stakeholders": "veterans and their families",
                "initiative": "educational programs about Malvinas"
            }
        elif event_type == "childrens_book_day":
            return {
                "finding": "reading rates are improving",
                "sample": "school districts nationwide",
                "result": "increased literacy engagement",
                "goal": "boost childhood literacy",
                "statistic": "1 in 5 children reading below grade level",
                "quote": "Every child deserves a book",
                "issue": "literacy gap",
                "source": "Department of Education",
                "action": "invest in libraries",
                "vision": "literate nation",
                "stakeholders": "educators, librarians, publishers",
                "initiative": "a $50 million literacy initiative"
            }
        elif event_type == "business_sakura":
            return {
                "finding": "strong economic momentum",
                "sample": "corporate earnings reports",
                "result": "positive market sentiment",
                "goal": "sustainable growth",
                "statistic": "2.5% projected GDP growth",
                "quote": "Japan's economy is resilient",
                "issue": "global economic uncertainty",
                "source": "Bank of Japan",
                "action": "strategic investment",
                "vision": "innovation-driven economy",
                "stakeholders": "business leaders and policymakers",
                "initiative": "corporate investment plans"
            }
        return {}

print("✅ HeadlineGenerator готов")

✅ HeadlineGenerator готов


In [10]:
# @title
# ============================================
# БЛОК 7: ОЦЕНКА ВЕРОЯТНОСТИ ПРЕДСКАЗАНИЙ
# ============================================

class ProbabilityScorer:
    """
    Класс для оценки вероятности и точности предсказаний
    """

    def __init__(self):
        pass

    def calculate_prediction_score(self, publication: Dict, event_type: str,
                                   country: str, shared_data: Dict,
                                   historical_articles_count: int) -> Dict:
        """
        Расчет комплексной оценки предсказания

        Returns:
            Словарь с оценками по разным критериям
        """
        # 1. Базовая вероятность издания (0-100)
        base_score = publication["likelihood"]

        # 2. Бонус за наличие праздников
        holiday_bonus = 0
        if shared_data.get("holidays"):
            holiday_bonus = min(len(shared_data["holidays"]) * 3, 15)

        # 3. Бонус за погодный контекст
        weather_bonus = 0
        if shared_data.get("weather") and shared_data["weather"].get("temperature"):
            weather_bonus = 5

        # 4. Бонус за исторические статьи
        historical_bonus = min(historical_articles_count / 10, 15)

        # 5. Бонус за тип издания
        type_bonus = {
            "medical_specialist": 10,
            "charity_specialist": 10,
            "official_agency": 10,
            "veteran_organization": 10,
            "library_specialist": 10,
            "publishing_trade": 8,
            "education_specialist": 8,
            "specialist_magazine": 8,
            "national_newspaper": 5,
            "public_broadcaster": 5,
            "regional": 3,
            "business_daily": 3,
            "general": 0
        }.get(publication["type"], 0)

        # 6. Бонус за соответствие специализации событию
        specialization_match = self._check_specialization_match(publication, event_type)

        # Итоговая оценка
        total_score = base_score + holiday_bonus + weather_bonus + historical_bonus + type_bonus + specialization_match

        # Ограничиваем 98%
        total_score = min(98, total_score)

        # Факторы, повлиявшие на оценку
        factors = []
        if base_score >= 90:
            factors.append(f"Высокая базовая вероятность издания ({base_score}%)")
        if holiday_bonus > 0:
            factors.append(f"Учтены праздники (+{holiday_bonus}%)")
        if weather_bonus > 0:
            factors.append(f"Погодный контекст (+{weather_bonus}%)")
        if historical_bonus > 0:
            factors.append(f"Анализ {historical_articles_count} исторических статей (+{historical_bonus:.0f}%)")
        if type_bonus > 0:
            factors.append(f"Специализированный тип издания (+{type_bonus}%)")
        if specialization_match > 0:
            factors.append(f"Соответствие специализации события (+{specialization_match}%)")

        # Определяем уровень уверенности
        if total_score >= 85:
            confidence = "Высокая"
            confidence_color = "green"
        elif total_score >= 70:
            confidence = "Средняя"
            confidence_color = "orange"
        else:
            confidence = "Низкая"
            confidence_color = "red"

        return {
            "total_score": total_score,
            "base_score": base_score,
            "holiday_bonus": holiday_bonus,
            "weather_bonus": weather_bonus,
            "historical_bonus": historical_bonus,
            "type_bonus": type_bonus,
            "specialization_match": specialization_match,
            "factors": factors,
            "confidence": confidence,
            "confidence_color": confidence_color
        }

    def _check_specialization_match(self, publication: Dict, event_type: str) -> int:
        """Проверка соответствия специализации издания событию"""
        specialization_map = {
            "autism": ["healthcare", "medical", "autism", "mental_health", "social_issues", "SEND"],
            "republic_day": ["politics", "official_news", "revolution", "military"],
            "malvinas_day": ["veterans", "military", "malvinas", "sovereignty", "politics"],
            "childrens_book_day": ["childrens_books", "literacy", "education", "libraries", "publishing"],
            "business_sakura": ["business", "economy", "travel", "tourism", "corporate"]
        }

        publication_specialization = publication.get("specialization", [])
        required_specializations = specialization_map.get(event_type, [])

        match_count = sum(1 for spec in publication_specialization if spec in required_specializations)

        if match_count >= 2:
            return 8
        elif match_count == 1:
            return 4
        return 0

    def rank_predictions(self, predictions: List[Dict]) -> List[Dict]:
        """Ранжирование предсказаний по вероятности"""
        sorted_predictions = sorted(predictions, key=lambda x: x["score"]["total_score"], reverse=True)

        for i, pred in enumerate(sorted_predictions, 1):
            pred["rank"] = i
            pred["rank_label"] = self._get_rank_label(i)

        return sorted_predictions

    def _get_rank_label(self, rank: int) -> str:
        """Получение метки ранга"""
        if rank == 1:
            return "🏆 НАИБОЛЕЕ ВЕРОЯТНОЕ"
        elif rank == 2:
            return "🥈 Второе по вероятности"
        elif rank == 3:
            return "🥉 Третье по вероятности"
        else:
            return f"{rank}-е по вероятности"

print("✅ ProbabilityScorer готов")

✅ ProbabilityScorer готов


In [40]:
# @title
# ============================================
# БЛОК 8.6: ИСПРАВЛЕННАЯ ВЕРСИЯ - ГЕНЕРАЦИЯ ПЕРВОГО АБЗАЦА
# ============================================

class FinalNewsPredictor:
    """
    Финальная версия с генерацией полноценного первого абзаца статьи
    """

    def __init__(self, gnews_api_key: str, calendar_api_key: str, weather_api_key: str):
        # Проверка API ключей
        if not gnews_api_key or gnews_api_key == "ВАШ_КЛЮЧ_ОТ_GNEWS":
            raise ValueError("❌ GNEWS_API_KEY обязателен! Получите на gnews.io")
        if not calendar_api_key or calendar_api_key == "ВАШ_КЛЮЧ_ОТ_CALENDARIFIC":
            raise ValueError("❌ CALENDAR_API_KEY обязателен! Получите на calendarific.com")
        if not weather_api_key or weather_api_key == "ВАШ_КЛЮЧ_ОТ_OPENWEATHER":
            raise ValueError("❌ WEATHER_API_KEY обязателен! Получите на openweathermap.org")

        self.gnews_api_key = gnews_api_key
        self.calendar_api_key = calendar_api_key
        self.weather_api_key = weather_api_key

        # Кэш
        self.cache = {}

        # Данные о странах
        self.countries_data = {
            "UK": {"code": "GB", "lang": "en", "capital": "London", "lat": 51.5074, "lon": -0.1278},
            "USA": {"code": "US", "lang": "en", "capital": "Washington, D.C.", "lat": 38.8951, "lon": -77.0364},
            "Iran": {"code": "IR", "lang": "fa", "capital": "Tehran", "lat": 35.6892, "lon": 51.3890},
            "Argentina": {"code": "AR", "lang": "es", "capital": "Buenos Aires", "lat": -34.6037, "lon": -58.3816},
            "Japan": {"code": "JP", "lang": "ja", "capital": "Tokyo", "lat": 35.6762, "lon": 139.6503}
        }

        self.flags = {"UK": "🇬🇧", "USA": "🇺🇸", "Iran": "🇮🇷", "Argentina": "🇦🇷", "Japan": "🇯🇵"}

        # URL API
        self.gnews_url = "https://gnews.io/api/v4"
        self.calendar_url = "https://calendarific.com/api/v2"
        self.weather_url = "https://api.openweathermap.org/data/2.5"

        # Упрощенные поисковые запросы
        self.search_queries = {
            "UK": {"autism": ["autism UK", "autism awareness UK"]},
            "USA": {"childrens_book_day": ["children's books USA", "literacy programs USA"]},
            "Japan": {"business_sakura": ["Japan economy", "Japan business news"]},
            "Argentina": {"malvinas_day": ["Malvinas Argentina", "veterans Argentina"]},
            "Iran": {"republic_day": ["Iran Republic Day", "Islamic Republic anniversary"]}
        }

        # Список изданий
        self.publications = {
            "UK": ["BBC News", "The Guardian", "The Times", "The Telegraph", "The Independent", "Financial Times"],
            "USA": ["CNN", "NPR", "The New York Times", "The Washington Post", "USA Today"],
            "Iran": ["Tehran Times", "Press TV", "IRNA", "Mehr News"],
            "Argentina": ["Clarín", "La Nación", "Infobae", "Página/12"],
            "Japan": ["NHK", "Nikkei", "Asahi Shimbun", "Yomiuri Shimbun"]
        }

    def _safe_extract(self, data, *keys, default=None):
        """Безопасное извлечение значения"""
        current = data
        for key in keys:
            if current is None:
                return default
            if isinstance(current, dict):
                current = current.get(key)
            elif isinstance(current, list):
                if len(current) > 0 and isinstance(current[0], dict):
                    current = current[0].get(key)
                else:
                    return default
            else:
                return default
        return current if current is not None else default

    def _make_request(self, url: str, params: Dict, retries: int = 3) -> Optional[Dict]:
        """Выполнение HTTP запроса"""
        for attempt in range(retries):
            try:
                response = requests.get(url, params=params, timeout=15)

                if response.status_code == 200:
                    data = response.json()
                    return data if isinstance(data, dict) else {"data": data}
                elif response.status_code == 429:
                    wait_time = 2 ** attempt
                    print(f"     ⏳ Лимит, ждем {wait_time} сек...")
                    time.sleep(wait_time)
                else:
                    return None
            except Exception as e:
                print(f"     ❌ Ошибка: {e}")
                if attempt < retries - 1:
                    time.sleep(2)
        return None

    def get_holidays(self, country: str, date: str) -> Dict[str, List[str]]:
        """Получение праздников"""
        cache_key = f"holidays_{country}_{date}"
        if cache_key in self.cache:
            return self.cache[cache_key]

        if country not in self.countries_data:
            return {"national": [], "global": []}

        country_data = self.countries_data[country]
        year, month, day = date.split('-')

        result = {"national": [], "global": []}

        params = {
            "api_key": self.calendar_api_key,
            "country": country_data["code"],
            "year": year,
            "month": month,
            "day": day
        }

        print(f"  🌐 Calendarific: праздники {country}")
        response = self._make_request(f"{self.calendar_url}/holidays", params)

        if response:
            holidays = self._safe_extract(response, "response", "holidays")
            if holidays and isinstance(holidays, list):
                for h in holidays:
                    if isinstance(h, dict):
                        name = h.get("name")
                        if name:
                            result["national"].append(name)
                print(f"  ✅ Найдено {len(result['national'])} праздников")

        self.cache[cache_key] = result
        return result

    def get_weather(self, country: str, date: str) -> Dict:
        """Получение погоды"""
        cache_key = f"weather_{country}_{date}"
        if cache_key in self.cache:
            return self.cache[cache_key]

        if country not in self.countries_data:
            return None

        country_data = self.countries_data[country]

        params = {
            "lat": country_data["lat"],
            "lon": country_data["lon"],
            "appid": self.weather_api_key,
            "units": "metric"
        }

        print(f"  🌐 OpenWeatherMap: погода в {country_data['capital']}")
        response = self._make_request(f"{self.weather_url}/weather", params)

        if response:
            main = response.get("main", {})
            weather_list = response.get("weather", [])
            result = {
                "temperature": main.get("temp"),
                "description": weather_list[0].get("description", "") if weather_list else "",
                "city": country_data["capital"]
            }
            print(f"  ✅ {result['temperature']}°C, {result['description']}")
            self.cache[cache_key] = result
            return result

        return None

    def search_articles(self, country: str, event_type: str, days_back: int = 90) -> List[Dict]:
        """Поиск статей"""
        cache_key = f"articles_{country}_{event_type}_{days_back}"
        if cache_key in self.cache:
            return self.cache[cache_key]

        country_data = self.countries_data[country]
        queries = self.search_queries.get(country, {}).get(event_type, [f"{event_type} {country}"])

        all_articles = []

        for query in queries[:2]:
            params = {
                "q": query,
                "token": self.gnews_api_key,
                "lang": country_data["lang"],
                "max": 30,
                "from": (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d"),
                "to": datetime.now().strftime("%Y-%m-%d"),
                "sortby": "relevance"
            }

            print(f"  🌐 GNews: '{query}'")
            response = self._make_request(f"{self.gnews_url}/search", params)

            if response:
                articles = response.get("articles", [])
                if articles:
                    print(f"     ✅ +{len(articles)} статей")
                    all_articles.extend(articles)

        # Удаляем дубликаты
        seen = set()
        unique = []
        for a in all_articles:
            url = a.get("url", "")
            if url and url not in seen:
                seen.add(url)
                unique.append(a)

        print(f"  📊 Всего: {len(unique)} статей")
        self.cache[cache_key] = unique
        return unique

    def analyze_articles(self, articles: List[Dict]) -> Dict:
        """Анализ статей"""
        if not articles:
            return {"has_data": False, "count": 0}

        from collections import Counter
        sources = [a.get("source", {}).get("name", "") for a in articles if a.get("source")]
        top_sources = Counter(sources).most_common(5)

        return {
            "has_data": True,
            "count": len(articles),
            "top_sources": top_sources
        }

    def generate_headline(self, articles: List[Dict], event_type: str, country: str,
                         holidays: List[str]) -> str:
        """Генерация заголовка"""
        if not articles:
            return None

        titles = [a.get("title", "") for a in articles if a.get("title")]
        if not titles:
            return None

        import random
        base = random.choice(titles)

        if len(base) > 100:
            base = base[:97] + "..."

        if holidays and not any(h.lower() in base.lower() for h in holidays[:1]):
            return f"{holidays[0]}: {base}"

        return base

    def generate_first_paragraph(self, articles: List[Dict], event_type: str, country: str,
                                 holidays: List[str], weather: Dict) -> str:
        """
        Генерация полноценного ПЕРВОГО АБЗАЦА (не просто description)
        Формат: Кто? Что? Где? Когда? + контекст
        """
        if not articles:
            return None

        import random

        # 1. Берем реальные описания из статей как основу
        descriptions = [a.get("description", "") for a in articles if a.get("description")]

        # 2. Берем заголовки для дополнительного контекста
        titles = [a.get("title", "") for a in articles if a.get("title")]

        # 3. Берем источники
        sources = [a.get("source", {}).get("name", "") for a in articles if a.get("source")]

        # Формируем первый абзац из нескольких частей

        # Часть 1: Введение (кто, что, где, когда)
        intro_templates = [
            f"In a significant development, {random.choice(sources) if sources else 'officials'} in {country} have announced",
            f"Today marks a pivotal moment as {country} unveils",
            f"According to {random.choice(sources) if sources else 'official sources'},",
            f"The government of {country} has today announced",
            f"In what experts call a groundbreaking move, {country} has introduced"
        ]

        # Часть 2: Основное содержание (из реальных статей)
        main_content = ""
        if descriptions:
            main_content = random.choice(descriptions)
            # Очищаем от лишних кавычек и точек
            main_content = main_content.strip()
            if main_content.endswith('.'):
                main_content = main_content[:-1]
        elif titles:
            main_content = random.choice(titles).strip()
        else:
            main_content = f"new initiatives related to {event_type}"

        # Часть 3: Контекст (праздники и погода)
        context_parts = []
        if holidays:
            context_parts.append(f"as the world marks {holidays[0]}")
        if weather and weather.get("temperature"):
            context_parts.append(f"with {weather['description']} and {weather['temperature']}°C in {weather['city']}")

        context = ""
        if context_parts:
            context = f", {', '.join(context_parts)}"

        # Часть 4: Завершение
        ending_templates = [
            f", according to reports from {random.choice(sources) if sources else 'local media'}.",
            f", marking a significant step forward.",
            f", officials confirmed during a press briefing.",
            f", as the nation observes this important day."
        ]

        # Собираем полноценный первый абзац
        intro = random.choice(intro_templates)
        ending = random.choice(ending_templates)

        # Формируем абзац
        if context:
            first_paragraph = f"{intro}{context} {main_content}{ending}"
        else:
            first_paragraph = f"{intro} {main_content}{ending}"

        # Убираем возможные повторяющиеся точки
        first_paragraph = first_paragraph.replace("..", ".")

        return first_paragraph

    def calculate_accuracy(self, articles: List[Dict], holidays: List[str], weather: Dict) -> float:
        """Расчет точности"""
        score = 50.0
        if articles:
            score += min(len(articles) / 2, 30)
        if holidays:
            score += min(len(holidays) * 5, 15)
        if weather:
            score += 5
        return min(98.0, score)

    def generate_predictions(self, country: str, event_type: str, num: int = 8) -> List[Dict]:
        """Генерация предсказаний"""
        print(f"\n{'='*60}")
        print(f"📊 {self.flags.get(country, '🌍')} {country} - {event_type}")
        print(f"{'='*60}")

        # Получаем данные
        holidays_data = self.get_holidays(country, "2026-04-02")
        holidays = holidays_data.get("national", []) + holidays_data.get("global", [])
        weather = self.get_weather(country, "2026-04-02")
        articles = self.search_articles(country, event_type, days_back=90)

        if not articles:
            print(f"  ❌ Нет статей")
            return []

        # Анализ
        analysis = self.analyze_articles(articles)
        print(f"\n  📊 {analysis['count']} статей")
        if analysis['top_sources']:
            print(f"  📰 Топ: {', '.join([s[0] for s in analysis['top_sources'][:3]])}")

        # Генерация предсказаний
        predictions = []
        top_sources = [s[0] for s in analysis['top_sources']]
        all_sources = top_sources + self.publications.get(country, [])
        unique_sources = list(dict.fromkeys(all_sources))[:num]

        for i, pub in enumerate(unique_sources, 1):
            print(f"\n  [{i}/{len(unique_sources)}] {pub}")

            # Фильтруем статьи из этого издания
            pub_articles = [a for a in articles if pub.lower() in a.get("source", {}).get("name", "").lower()]
            use_articles = pub_articles if pub_articles else articles
            count = len(pub_articles) if pub_articles else len(articles)

            headline = self.generate_headline(use_articles, event_type, country, holidays)
            first_paragraph = self.generate_first_paragraph(use_articles, event_type, country, holidays, weather)
            accuracy = self.calculate_accuracy(use_articles, holidays, weather)

            if headline and first_paragraph:
                predictions.append({
                    "country": country,
                    "country_flag": self.flags.get(country, "🌍"),
                    "event_type": event_type,
                    "publication": pub,
                    "articles_analyzed": count,
                    "headline": headline,
                    "first_paragraph": first_paragraph,
                    "accuracy_score": accuracy,
                    "holidays": holidays,
                    "weather": weather
                })
                print(f"     ✅ Точность: {accuracy:.0f}%")
            else:
                print(f"     ❌ Ошибка")

        # Сортировка
        predictions.sort(key=lambda x: x["accuracy_score"], reverse=True)
        for i, p in enumerate(predictions, 1):
            p["rank"] = i
            labels = {1: "🏆 НАИБОЛЕЕ ВЕРОЯТНОЕ", 2: "🥈 Второе", 3: "🥉 Третье"}
            p["rank_label"] = labels.get(i, f"{i}-е")

        return predictions

    def generate_all(self, countries_events: List[Tuple[str, str]], num: int = 8) -> Dict:
        """Генерация для всех стран"""
        results = {}
        for country, event in countries_events:
            try:
                preds = self.generate_predictions(country, event, num)
                if preds:
                    results[f"{country}_{event}"] = preds
            except Exception as e:
                print(f"  ❌ Ошибка {country}: {e}")
        return results

print("✅ FinalNewsPredictor готов (генерирует полноценный первый абзац)")

✅ FinalNewsPredictor готов (генерирует полноценный первый абзац)


In [41]:
# @title
# ============================================
# БЛОК 9: ЗАПУСК ФИНАЛЬНОЙ ВЕРСИИ
# ============================================

# ВСТАВЬТЕ ВАШИ API КЛЮЧИ
GNEWS_API_KEY = "92cea5faf7800eaa4302c868bcb934ec"
CALENDAR_API_KEY = "uLWjql0491TAhgt77einRqPhNSopFWqW"
WEATHER_API_KEY = "39c76ed23fa5174e59309ef9b6f3fb98"

print("="*80)
print("🚀 ЗАПУСК УПРОЩЕННОЙ ВЕРСИИ")
print("="*80)

# Проверка
if GNEWS_API_KEY == "ВАШ_КЛЮЧ_ОТ_GNEWS":
    print("❌ Получите GNews ключ: https://gnews.io/")
elif CALENDAR_API_KEY == "ВАШ_КЛЮЧ_ОТ_CALENDARIFIC":
    print("❌ Получите Calendarific ключ: https://calendarific.com/")
elif WEATHER_API_KEY == "ВАШ_КЛЮЧ_ОТ_OPENWEATHER":
    print("❌ Получите OpenWeatherMap ключ: https://openweathermap.org/")
else:
    predictor = SimplifiedNewsPredictor(GNEWS_API_KEY, CALENDAR_API_KEY, WEATHER_API_KEY)

    countries_events = [
        ("UK", "autism"),
        ("USA", "childrens_book_day"),
        ("Japan", "business_sakura"),
        ("Argentina", "malvinas_day"),
        ("Iran", "republic_day")
    ]

    results = predictor.generate_all(countries_events, num=8)

    print("\n" + "="*80)
    print("📊 ИТОГОВАЯ СТАТИСТИКА")
    print("="*80)

    total = 0
    for key, preds in results.items():
        country = key.split("_")[0]
        print(f"  • {country}: {len(preds)} предсказаний")
        total += len(preds)

    print(f"\n  ✅ ВСЕГО: {total} предсказаний")

🚀 ЗАПУСК УПРОЩЕННОЙ ВЕРСИИ

📊 🇬🇧 UK - autism
  🌐 Calendarific: праздники UK
  ✅ Найдено 2 праздников
  🌐 OpenWeatherMap: погода в London
  ✅ 8.45°C, clear sky
  🌐 GNews: 'autism UK'
  🌐 GNews: 'autism awareness UK'
  📊 Всего: 0 уникальных статей
  ❌ Нет статей

📊 🇺🇸 USA - childrens_book_day
  🌐 Calendarific: праздники USA
  ✅ Найдено 4 праздников
  🌐 OpenWeatherMap: погода в Washington, D.C.
  ✅ 20.38°C, overcast clouds
  🌐 GNews: 'children's books'
     ❌ Ошибка 400: неверный запрос
  🌐 GNews: 'literacy USA'
  📊 Всего: 0 уникальных статей
  ❌ Нет статей

📊 🇯🇵 Japan - business_sakura
  🌐 Calendarific: праздники Japan
  🌐 OpenWeatherMap: погода в Tokyo
  ✅ 16.67°C, overcast clouds
  🌐 GNews: 'Japan economy'
     ⏳ Лимит, ждем 1 сек...
  🌐 GNews: 'Japan business'
     ✅ +3 статей
  📊 Всего: 3 уникальных статей

  📊 3 статей, тональность: neutral
  📰 Топ: Business Insider Japan

  [1/6] Business Insider Japan
     ✅ Точность: 56%

  [2/6] NHK
     ✅ Точность: 56%

  [3/6] Nikkei
     ✅ То

In [42]:
# @title
# ============================================
# БЛОК 10: ОТОБРАЖЕНИЕ РЕЗУЛЬТАТОВ
# ============================================

from IPython.display import display, HTML
import pandas as pd

def display_prediction(prediction: Dict, idx: int = None):
    """
    Красивое отображение одного предсказания
    """
    accuracy = prediction["accuracy_score"]

    if accuracy >= 85:
        accuracy_color = "#4CAF50"
        confidence_label = "Высокая"
    elif accuracy >= 70:
        accuracy_color = "#FF9800"
        confidence_label = "Средняя"
    else:
        accuracy_color = "#f44336"
        confidence_label = "Низкая"

    flag = prediction.get("country_flag", "🌍")
    country = prediction["country"]
    event = prediction["event_type"]
    publication = prediction["publication"]
    articles_count = prediction["articles_analyzed"]
    rank_label = prediction.get("rank_label", "")

    # Погодная информация
    weather = prediction.get("weather", {})
    weather_html = ""
    if weather and weather.get("temperature"):
        weather_html = f"""
        <div style="background: #e8f0fe; padding: 5px 12px; border-radius: 20px; display: inline-block; margin-right: 8px;">
            🌡️ {weather.get('city', country)}: {weather['temperature']}°C, {weather.get('description', '')}
        </div>
        """

    # Праздники
    holidays = prediction.get("holidays", [])
    holidays_html = ""
    if holidays:
        holidays_html = f"""
        <div style="background: #fff3e0; padding: 5px 12px; border-radius: 20px; display: inline-block;">
            📅 {', '.join(holidays[:2])}
        </div>
        """

    # Ранг
    rank_html = ""
    if rank_label:
        rank_bg = {"🏆": "#FFD700", "🥈": "#C0C0C0", "🥉": "#CD7F32"}.get(rank_label[:2], "#e0e0e0")
        rank_html = f"""
        <div style="background: {rank_bg}; padding: 5px 12px; border-radius: 20px; display: inline-block; font-weight: bold;">
            {rank_label}
        </div>
        """

    html = f"""
    <div style="margin: 20px 0; padding: 20px; border-left: 5px solid {accuracy_color};
                background: #f9f9f9; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">

        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px; flex-wrap: wrap; gap: 10px;">
            <div>
                <span style="font-size: 1.8em;">{flag}</span>
                <strong style="font-size: 1.3em; margin-left: 8px;">{country}</strong>
                <span style="margin-left: 12px; color: #666; background: #f0f0f0; padding: 3px 10px; border-radius: 20px;">
                    {event}
                </span>
            </div>
            <div style="display: flex; gap: 8px;">
                {rank_html}
                <div style="background: {accuracy_color}; color: white; padding: 5px 15px; border-radius: 25px; font-weight: bold;">
                    🎯 {accuracy:.1f}% | {confidence_label}
                </div>
            </div>
        </div>

        <div style="background: white; padding: 15px; border-radius: 10px; margin-bottom: 15px; border: 1px solid #e0e0e0;">
            <div style="display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap;">
                <strong style="font-size: 1.1em;">📰 {publication}</strong>
                <span style="color: #666; font-size: 0.85em; background: #f5f5f5; padding: 3px 10px; border-radius: 15px;">
                    📊 {articles_count} статей проанализировано
                </span>
            </div>
        </div>

        <div style="background: #e3f2fd; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
            <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 10px;">
                <span style="font-size: 1.2em;">📢</span>
                <strong style="color: #1976d2;">ЗАГОЛОВОК</strong>
            </div>
            <div style="font-size: 1.1em; font-weight: bold; line-height: 1.4;">
                "{prediction['headline']}"
            </div>
        </div>

        <div style="background: #fff3e0; padding: 15px; border-radius: 10px; margin-bottom: 15px;">
            <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 10px;">
                <span style="font-size: 1.2em;">📝</span>
                <strong style="color: #f57c00;">ПЕРВЫЙ АБЗАЦ</strong>
            </div>
            <div style="line-height: 1.5;">
                {prediction['first_paragraph']}
            </div>
        </div>

        <div style="display: flex; flex-wrap: wrap; gap: 10px; margin-top: 10px; padding-top: 10px; border-top: 1px solid #e0e0e0;">
            {weather_html}
            {holidays_html}
        </div>
    </div>
    """

    display(HTML(html))


def display_all_results(results: Dict[str, List[Dict]]):
    """
    Отображение всех результатов с красивым оформлением
    """
    print("\n" + "="*80)
    print("📰 ПРЕДСКАЗАНИЯ НОВОСТЕЙ НА 2 АПРЕЛЯ 2026")
    print("="*80)
    print("📌 Источники: реальные данные из GNews API, Calendarific API, OpenWeatherMap API")
    print("="*80)

    total_predictions = 0

    for key, predictions in results.items():
        if not predictions:
            continue

        country = predictions[0]["country"]
        event = predictions[0]["event_type"]

        print(f"\n{'='*60}")
        print(f"🌍 {country} - {event}")
        print(f"{'='*60}")

        # Показываем топ-5 предсказаний
        for i, pred in enumerate(predictions[:5], 1):
            print(f"\n--- #{i}: {pred['publication']} (точность: {pred['accuracy_score']:.0f}%) ---")
            display_prediction(pred, i)

        total_predictions += len(predictions)

        # Статистика по стране
        avg_accuracy = sum(p["accuracy_score"] for p in predictions) / len(predictions)
        print(f"\n📊 Статистика по {country}:")
        print(f"  • Всего предсказаний: {len(predictions)}")
        print(f"  • Средняя точность: {avg_accuracy:.1f}%")
        print(f"  • Лучшая точность: {predictions[0]['accuracy_score']:.1f}% ({predictions[0]['publication']})")
        print(f"  • Всего проанализировано статей: {sum(p['articles_analyzed'] for p in predictions)}")

    print(f"\n{'='*80}")
    print(f"✅ ВСЕГО СГЕНЕРИРОВАНО: {total_predictions} ПРЕДСКАЗАНИЙ")
    print(f"{'='*80}")

print("✅ Функции отображения готовы")

✅ Функции отображения готовы


In [43]:
# @title
# ============================================
# БЛОК 11: СОХРАНЕНИЕ РЕЗУЛЬТАТОВ В CSV И EXCEL
# ============================================

def save_results_to_csv(results: Dict[str, List[Dict]], filename: str = None):
    """
    Сохранение результатов в CSV файл
    """
    from datetime import datetime

    data = []
    for key, predictions in results.items():
        for pred in predictions:
            data.append({
                "Страна": pred["country"],
                "Событие": pred["event_type"],
                "Ранг": pred.get("rank", ""),
                "Ранг_метка": pred.get("rank_label", ""),
                "Издание": pred["publication"],
                "Проанализировано_статей": pred["articles_analyzed"],
                "Точность_%": round(pred["accuracy_score"], 1),
                "Заголовок": pred["headline"],
                "Первый_абзац": pred["first_paragraph"],
                "Праздники": ", ".join(pred.get("holidays", [])),
                "Погода": f"{pred.get('weather', {}).get('temperature', 'N/A')}°C, {pred.get('weather', {}).get('description', 'N/A')}",
                "Город": pred.get('weather', {}).get('city', 'N/A')
            })

    df = pd.DataFrame(data)

    if filename is None:
        filename = f"news_predictions_2026_04_02_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n💾 Результаты сохранены в CSV: {filename}")

    return df


def save_results_to_excel(results: Dict[str, List[Dict]], filename: str = None):
    """
    Сохранение результатов в Excel файл с форматированием
    """
    from datetime import datetime

    data = []
    for key, predictions in results.items():
        for pred in predictions:
            data.append({
                "Страна": pred["country"],
                "Событие": pred["event_type"],
                "Ранг": pred.get("rank", ""),
                "Ранг_метка": pred.get("rank_label", ""),
                "Издание": pred["publication"],
                "Проанализировано_статей": pred["articles_analyzed"],
                "Точность_%": round(pred["accuracy_score"], 1),
                "Заголовок": pred["headline"],
                "Первый_абзац": pred["first_paragraph"],
                "Праздники": ", ".join(pred.get("holidays", [])),
                "Погода": f"{pred.get('weather', {}).get('temperature', 'N/A')}°C, {pred.get('weather', {}).get('description', 'N/A')}",
                "Город": pred.get('weather', {}).get('city', 'N/A')
            })

    df = pd.DataFrame(data)

    if filename is None:
        filename = f"news_predictions_2026_04_02_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

    # Создаем Excel файл с форматированием
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='News Predictions', index=False)

        # Получаем рабочую книгу и лист
        workbook = writer.book
        worksheet = writer.sheets['News Predictions']

        # Настраиваем ширину колонок
        column_widths = {
            'A': 12,  # Страна
            'B': 18,  # Событие
            'C': 8,   # Ранг
            'D': 22,  # Ранг_метка
            'E': 30,  # Издание
            'F': 18,  # Проанализировано_статей
            'G': 12,  # Точность_%
            'H': 50,  # Заголовок
            'I': 80,  # Первый_абзац
            'J': 40,  # Праздники
            'K': 25,  # Погода
            'L': 15   # Город
        }

        for col, width in column_widths.items():
            worksheet.column_dimensions[col].width = width

        # Добавляем условное форматирование для точности
        from openpyxl.styles import PatternFill
        from openpyxl.formatting.rule import Rule
        from openpyxl.styles.differential import DifferentialStyle

        # Зеленый для высокой точности (>85%)
        green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
        rule_green = Rule(type="expression", dxf=DifferentialStyle(fill=green_fill), formula=["$G2>85"])
        worksheet.conditional_formatting.add("G2:G1000", rule_green)

        # Оранжевый для средней точности (70-85%)
        orange_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
        rule_orange = Rule(type="expression", dxf=DifferentialStyle(fill=orange_fill), formula=["AND($G2>=70, $G2<=85)"])
        worksheet.conditional_formatting.add("G2:G1000", rule_orange)

        # Красный для низкой точности (<70%)
        red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
        rule_red = Rule(type="expression", dxf=DifferentialStyle(fill=red_fill), formula=["$G2<70"])
        worksheet.conditional_formatting.add("G2:G1000", rule_red)

    print(f"\n💾 Результаты сохранены в Excel: {filename}")

    return df


def print_summary_statistics(results: Dict[str, List[Dict]]):
    """
    Вывод сводной статистики по всем предсказаниям
    """
    print("\n" + "="*80)
    print("📊 СВОДНАЯ СТАТИСТИКА")
    print("="*80)

    all_predictions = []
    for predictions in results.values():
        all_predictions.extend(predictions)

    if not all_predictions:
        print("⚠️ Нет данных для статистики")
        return

    # Общая статистика
    print(f"\n📈 ОБЩАЯ СТАТИСТИКА:")
    print(f"  • Всего предсказаний: {len(all_predictions)}")
    print(f"  • Средняя точность: {sum(p['accuracy_score'] for p in all_predictions) / len(all_predictions):.1f}%")
    print(f"  • Максимальная точность: {max(p['accuracy_score'] for p in all_predictions):.1f}%")
    print(f"  • Минимальная точность: {min(p['accuracy_score'] for p in all_predictions):.1f}%")

    # Статистика по странам
    print(f"\n🌍 СТАТИСТИКА ПО СТРАНАМ:")
    for country in set(p["country"] for p in all_predictions):
        country_preds = [p for p in all_predictions if p["country"] == country]
        avg_acc = sum(p["accuracy_score"] for p in country_preds) / len(country_preds)
        print(f"  • {country}: {len(country_preds)} предсказаний, средняя точность: {avg_acc:.1f}%")

    # Статистика по типам изданий (топ-5)
    print(f"\n📰 ТОП-5 ИЗДАНИЙ ПО ЧАСТОТЕ ПОЯВЛЕНИЯ:")
    from collections import Counter
    pub_counter = Counter(p["publication"] for p in all_predictions)
    for pub, count in pub_counter.most_common(5):
        avg_acc = sum(p["accuracy_score"] for p in all_predictions if p["publication"] == pub) / count
        print(f"  • {pub}: {count} раз, средняя точность: {avg_acc:.1f}%")

    # Распределение точности
    print(f"\n📊 РАСПРЕДЕЛЕНИЕ ПО УРОВНЯМ ТОЧНОСТИ:")
    high = sum(1 for p in all_predictions if p["accuracy_score"] >= 85)
    medium = sum(1 for p in all_predictions if 70 <= p["accuracy_score"] < 85)
    low = sum(1 for p in all_predictions if p["accuracy_score"] < 70)

    print(f"  • Высокая точность (≥85%): {high} ({high/len(all_predictions)*100:.1f}%)")
    print(f"  • Средняя точность (70-84%): {medium} ({medium/len(all_predictions)*100:.1f}%)")
    print(f"  • Низкая точность (<70%): {low} ({low/len(all_predictions)*100:.1f}%)")

print("✅ Функции сохранения готовы")

✅ Функции сохранения готовы


In [44]:
# @title
# ============================================
# БЛОК 12: ВЫПОЛНЕНИЕ ВСЕХ ОПЕРАЦИЙ
# ============================================

# Проверяем, что результаты получены
if 'results' in dir() and results:
    print("\n" + "="*80)
    print("🎯 ОБРАБОТКА РЕЗУЛЬТАТОВ")
    print("="*80)

    # 1. Отображаем результаты
    display_all_results(results)

    # 2. Сохраняем в CSV
    df_csv = save_results_to_csv(results)

    # 3. Сохраняем в Excel (требуется openpyxl)
    try:
        import openpyxl
        df_excel = save_results_to_excel(results)
    except ImportError:
        print("\n⚠️ Для сохранения в Excel установите openpyxl: !pip install openpyxl")

    # 4. Выводим сводную статистику
    print_summary_statistics(results)

    # 5. Показываем первые строки таблицы
    print("\n" + "="*80)
    print("📋 ПРЕДПРОСМОТР ТАБЛИЦЫ РЕЗУЛЬТАТОВ")
    print("="*80)
    display(df_csv.head(10))

    print("\n" + "="*80)
    print("✅ ВСЕ ОПЕРАЦИИ УСПЕШНО ЗАВЕРШЕНЫ")
    print("="*80)

else:
    print("\n⚠️ Нет результатов для отображения. Проверьте:")
    print("  1. API ключи (GNews, Calendarific, OpenWeatherMap)")
    print("  2. Интернет-соединение")
    print("  3. Лимиты запросов API")


🎯 ОБРАБОТКА РЕЗУЛЬТАТОВ

📰 ПРЕДСКАЗАНИЯ НОВОСТЕЙ НА 2 АПРЕЛЯ 2026
📌 Источники: реальные данные из GNews API, Calendarific API, OpenWeatherMap API

🌍 Japan - business_sakura

--- #1: Business Insider Japan (точность: 56%) ---



--- #2: NHK (точность: 56%) ---



--- #3: Nikkei (точность: 56%) ---



--- #4: Asahi Shimbun (точность: 56%) ---



--- #5: Yomiuri Shimbun (точность: 56%) ---


...


--- #4: Firstpost (точность: 56%) ---



--- #5: NewsX (точность: 56%) ---



📊 Статистика по Iran:
  • Всего предсказаний: 8
  • Средняя точность: 56.4%
  • Лучшая точность: 58.0% (Tehran Times)
  • Всего проанализировано статей: 23

✅ ВСЕГО СГЕНЕРИРОВАНО: 14 ПРЕДСКАЗАНИЙ

💾 Результаты сохранены в CSV: news_predictions_2026_04_02_20260330_205439.csv

💾 Результаты сохранены в Excel: news_predictions_2026_04_02_20260330_205439.xlsx

📊 СВОДНАЯ СТАТИСТИКА

📈 ОБЩАЯ СТАТИСТИКА:
  • Всего предсказаний: 14
  • Средняя точность: 56.5%
  • Максимальная точность: 58.0%
  • Минимальная точность: 55.5%

🌍 СТАТИСТИКА ПО СТРАНАМ:
  • Japan: 6 предсказаний, средняя точность: 56.5%
  • Iran: 8 предсказаний, средняя точность: 56.4%

📰 ТОП-5 ИЗДАНИЙ ПО ЧАСТОТЕ ПОЯВЛЕНИЯ:
  • Business Insider Japan: 1 раз, средняя точность: 56.5%
  • NHK: 1 раз, средняя точность: 56.5%
  • Nikkei: 1 раз, средняя точность: 56.5%
  • Asahi Shimbun: 1 раз, средняя точность: 56.5%
  • Yomiuri Shimbun: 1 раз, средняя точность: 56.5%

📊 РАСПРЕДЕЛЕНИЕ ПО УРОВНЯМ ТОЧНОСТИ:
  • Высокая точность (≥85%): 0 

,Страна,Событие,Ранг,Ранг_метка,Издание,Проанализировано_статей,Точность_%,Заголовок,Первый_абзац,Праздники,Погода,Город
0,Japan,business_sakura,1,🏆 НАИБОЛЕЕ ВЕРОЯТНОЕ,Business Insider Japan,3,56.5,カレンダーアプリを実際に作ってみた。AIに話しかけるだけでアプリが作れる「バイブコーディング」とは,with overcast clouds and 16.67°C in Tokyo. AIに...,,"16.67°C, overcast clouds",Tokyo
1,Japan,business_sakura,2,🥈 Второе,NHK,3,56.5,BI PREMIUM 料金改定、および「PREMIUMニュースレター」4月開始のおしらせ,with overcast clouds and 16.67°C in Tokyo. AIに...,,"16.67°C, overcast clouds",Tokyo
2,Japan,business_sakura,3,🥉 Третье,Nikkei,3,56.5,職場は変わった、社会は──？ 10代から60代、800人の「正直な答え」が示すジェンダーの現在地,with overcast clouds and 16.67°C in Tokyo. AIに...,,"16.67°C, overcast clouds",Tokyo
3,Japan,business_sakura,4,4-е,Asahi Shimbun,3,56.5,カレンダーアプリを実際に作ってみた。AIに話しかけるだけでアプリが作れる「バイブコーディング」とは,with overcast clouds and 16.67°C in Tokyo. AIに...,,"16.67°C, overcast clouds",Tokyo
4,Japan,business_sakura,5,5-е,Yomiuri Shimbun,3,56.5,BI PREMIUM 料金改定、および「PREMIUMニュースレター」4月開始のおしらせ,with overcast clouds and 16.67°C in Tokyo. AIに...,,"16.67°C, overcast clouds",Tokyo
5,Japan,business_sakura,6,6-е,Mainichi Shimbun,3,56.5,職場は変わった、社会は──？ 10代から60代、800人の「正直な答え」が示すジェンダーの現在地,with overcast clouds and 16.67°C in Tokyo. AIに...,,"16.67°C, overcast clouds",Tokyo
6,Iran,republic_day,1,🏆 НАИБОЛЕЕ ВЕРОЯТНОЕ,Tehran Times,6,58.0,No Fuel For Patriotism: Pakistan Cancels Repub...,with light rain and 13.48°C in Tehran. The rev...,,"13.48°C, light rain",Tehran
7,Iran,republic_day,2,🥈 Второе,Press TV,6,58.0,Israel hits Tehran as Iran's counterattacks wi...,with light rain and 13.48°C in Tehran. Pakista...,,"13.48°C, light rain",Tehran
8,Iran,republic_day,3,🥉 Третье,IRNA,6,58.0,The Latest: Israel hits Tehran as Iran’s count...,with light rain and 13.48°C in Tehran. Pakista...,,"13.48°C, light rain",Tehran
9,Iran,republic_day,4,4-е,Firstpost,1,55.5,Why has Pakistan called off its Republic Day p...,with light rain and 13.48°C in Tehran. Pakista...,,"13.48°C, light rain",Tehran



✅ ВСЕ ОПЕРАЦИИ УСПЕШНО ЗАВЕРШЕНЫ
